In [1]:
# --- YOUR PATHS (Windows raw strings) ---
BASE_CORRIDOR = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\3_Corridor\3_Corridor"
BASE_NH      = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\MyNursingHome\MyNursingHome\MYNursingHome\MYNursingHome"

# sample files you showed
SAMPLE_CORRIDOR_IMG = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\3_Corridor\3_Corridor\exp400\exp400_img62.jpg"
SAMPLE_NH_IMG       = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\MyNursingHome\MyNursingHome\MYNursingHome\MYNursingHome\basket_bin\00001.jpg"

# outputs under your desktop project
OUT_ROOT   = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs"
DEPTH_DIR  = rf"{OUT_ROOT}\depth_maps"
GRID_DIR   = rf"{OUT_ROOT}\occupancy_grids"
VID_DIR    = rf"{OUT_ROOT}\videos"
FRAME_DIR  = rf"{OUT_ROOT}\frames"

import os
for d in [OUT_ROOT, DEPTH_DIR, GRID_DIR, VID_DIR, FRAME_DIR]:
    os.makedirs(d, exist_ok=True)


In [2]:
!pip install ultralytics opencv-python transformers timm torch torchvision numpy --quiet


In [8]:
import os, re, glob, cv2, numpy as np, torch, heapq, time
from ultralytics import YOLO
from transformers import DPTImageProcessor, DPTForDepthEstimation

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cpu


In [4]:
# Tiny/fast YOLO; you can swap to your own weights later
yolo = YOLO("yolov8n.pt")

# MiDaS (DPT-hybrid) for monocular depth
depth_model = DPTForDepthEstimation.from_pretrained("Intel/dpt-hybrid-midas").to(device)
depth_proc  = DPTImageProcessor.from_pretrained("Intel/dpt-hybrid-midas")


In [6]:
def test_on_image(img_path, out_name):
    img = cv2.imread(img_path)
    assert img is not None, f"Can't read: {img_path}"
    boxes = yolo_obstacles(img, conf=0.25)
    depth = estimate_depth(img)
    mask  = boxes_to_mask(img.shape, boxes, inflate=12)
    grid  = build_occupancy(depth, mask, depth_thresh=165, dilate_iter=2)

    H,W = grid.shape
    start = (int(H*0.85), int(W*0.5))
    goal  = (int(H*0.15), int(W*0.5))
    path  = astar(grid, start, goal)

    vis = img.copy()
    for (x1,y1,x2,y2) in boxes:
        cv2.rectangle(vis,(x1,y1),(x2,y2),(0,0,255),2)
    for (r,c) in path:
        cv2.circle(vis,(c,r),2,(0,255,0),-1)

    cv2.imwrite(os.path.join(FRAME_DIR, out_name), vis)
    print(f"saved: {os.path.join(FRAME_DIR, out_name)}  | boxes={len(boxes)} path_len={len(path)}")

# run on your two samples
test_on_image(SAMPLE_CORRIDOR_IMG, "corridor_sample_overlay.jpg")
test_on_image(SAMPLE_NH_IMG,       "mynursinghome_sample_overlay.jpg")


saved: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\frames\corridor_sample_overlay.jpg  | boxes=0 path_len=253
saved: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\frames\mynursinghome_sample_overlay.jpg  | boxes=0 path_len=412


In [9]:
import time, glob, os, numpy as np, cv2

# pick a short subset from one sequence (e.g., exp400)
SAMPLE_SEQ = rf"{BASE_CORRIDOR}\exp400"
sample_frames = sorted(glob.glob(rf"{SAMPLE_SEQ}\exp400_img*.jpg"))[:30]  # first 30 frames

REPLAN_EVERY = 5  # how often to re-run A* (you can change this)

def profile_pipeline(frames, replan_every=5):
    t_det = []    # YOLO detection time
    t_depth = []  # MiDaS depth time
    t_grid = []   # Occupancy grid time
    t_plan = []   # Path planning time
    t_total = []  # Total per-frame time

    last_path = []

    # optional warm-up to load models
    if frames:
        _ = yolo_obstacles(cv2.imread(frames[0]), conf=0.25)
        _ = estimate_depth(cv2.imread(frames[0]))

    for i, fp in enumerate(frames, 1):
        frame = cv2.imread(fp)
        H, W = frame.shape[:2]

        t0 = time.time()

        # YOLO detection
        t1 = time.time()
        boxes = yolo_obstacles(frame, conf=0.25)
        t_det.append(time.time() - t1)

        # Depth estimation
        t2 = time.time()
        depth = estimate_depth(frame)
        t_depth.append(time.time() - t2)

        # Occupancy grid
        mask = boxes_to_mask(frame.shape, boxes, inflate=12)
        t3 = time.time()
        grid = build_occupancy(depth, mask, depth_thresh=165, dilate_iter=2)
        t_grid.append(time.time() - t3)

        # A* path planning (every few frames)
        start = (int(H * 0.85), int(W * 0.5))
        goal  = (int(H * 0.15), int(W * 0.5))
        if not last_path or (i % replan_every == 0):
            t4 = time.time()
            last_path = astar(grid, start, goal)
            t_plan.append(time.time() - t4)
        else:
            t_plan.append(0.0)

        t_total.append(time.time() - t0)

    # compute averages
    def avg(x): return (sum(x) / max(1, len(x)))

    print(f"Avg per frame (sec):")
    print(f"  detection = {avg(t_det):.3f}")
    print(f"  depth     = {avg(t_depth):.3f}")
    print(f"  occupancy = {avg(t_grid):.3f}")
    print(f"  planning  = {avg(t_plan):.3f}")
    print(f"  total     = {avg(t_total):.3f}")
    print(f"Approx FPS  = {1.0 / max(1e-9, avg(t_total)):.2f}")

    return {
        "detect_s": np.mean(t_det),
        "depth_s": np.mean(t_depth),
        "grid_s": np.mean(t_grid),
        "plan_s": np.mean(t_plan),
        "total_s": np.mean(t_total)
    }

# run the profiler
profile = profile_pipeline(sample_frames, replan_every=REPLAN_EVERY)


Avg per frame (sec):
  detection = 0.126
  depth     = 2.774
  occupancy = 0.002
  planning  = 0.001
  total     = 2.903
Approx FPS  = 0.34


In [6]:
# --- define base paths ---
BASE_CORRIDOR = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\3_Corridor\3_Corridor"
VID_DIR = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos"

import os
os.makedirs(VID_DIR, exist_ok=True)


In [35]:
# === imports ===
import os, re, glob, cv2, heapq, torch, numpy as np
from ultralytics import YOLO
from transformers import DPTForDepthEstimation, DPTImageProcessor

# === paths ===
BASE_CORRIDOR = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\3_Corridor\3_Corridor"
VID_DIR       = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos"
os.makedirs(VID_DIR, exist_ok=True)

# === device & models ===
device = "cuda" if torch.cuda.is_available() else "cpu"
yolo = YOLO("yolov8n.pt")

# === knobs ===
EXP_START, EXP_END = 400, 733
TEST_ONLY          = False          # set True to run a quick subset
INFLATE_PIXELS     = 8
REPLAN_EVERY       = 5

# YOLO tuning
DET_CONF       = 0.45
MIN_AREA_FRAC  = 0.002
MAX_AREA_FRAC  = 0.35
ALLOWED_CLASSES = {  # comment out to allow all
    "person","chair","bench","bed","couch","tv","tvmonitor","dining table",
    "potted plant","backpack","handbag","suitcase","bottle","fire hydrant","stop sign"
}

# Adaptive center-ROI depth probe
USE_DEPTH_PROBE  = True
PROBE_EVERY      = 6
PROBE_WIDTH_FRAC = 0.30
PROBE_TARGET_W   = 400

_depth_ready = False
def _ensure_depth_model():
    global _depth_ready, depth_model, depth_proc
    if _depth_ready: return
    depth_model = DPTForDepthEstimation.from_pretrained("Intel/dpt-hybrid-midas").to(device)
    depth_proc  = DPTImageProcessor.from_pretrained("Intel/dpt-hybrid-midas")
    _depth_ready = True

def depth_probe_mask(frame_bgr):
    """adaptive depth threshold via percentile; nearer=bright"""
    _ensure_depth_model()
    H, W = frame_bgr.shape[:2]
    cx1 = int(W*(0.5 - PROBE_WIDTH_FRAC/2)); cx2 = int(W*(0.5 + PROBE_WIDTH_FRAC/2))
    roi = frame_bgr[:, cx1:cx2]
    h, w = roi.shape[:2]
    tw = PROBE_TARGET_W; th = int(h * (tw/max(1,w)))
    roi_small = cv2.resize(roi, (tw, th))
    rgb = cv2.cvtColor(roi_small, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        inputs = depth_proc(images=rgb, return_tensors="pt").to(device)
        d = depth_model(**inputs).predicted_depth.squeeze().detach().cpu().numpy()
    d = (d - d.min()) / (d.max() - d.min() + 1e-8)
    d8 = 255 - (d*255).astype(np.uint8)  # nearer = bright
    d8 = cv2.resize(d8, (cx2-cx1, H), interpolation=cv2.INTER_LINEAR)
    thr = np.percentile(d8, 80)          # adaptive
    mask = (d8 >= thr).astype(np.uint8)*255
    full = np.zeros((H,W), np.uint8); full[:, cx1:cx2] = mask
    full = cv2.dilate(full, np.ones((3,3),np.uint8), iterations=3)
    full = cv2.GaussianBlur(full, (5,5), 0)
    _, full = cv2.threshold(full, 40, 255, cv2.THRESH_BINARY)
    return full

# --- helpers ---
def yolo_obstacles(frame_bgr, conf=DET_CONF):
    res = yolo.predict(source=frame_bgr, conf=conf, verbose=False,
                       device=0 if device=="cuda" else None)[0]
    H, W = frame_bgr.shape[:2]; A = H*W
    boxes = []
    if res.boxes is not None:
        xyxy = res.boxes.xyxy.cpu().numpy().astype(int)
        cls  = res.boxes.cls.cpu().numpy().astype(int) if res.boxes.cls is not None else None
        names= getattr(yolo.model, "names", None)
        for k,(x1,y1,x2,y2) in enumerate(xyxy):
            area_frac = max(1,(x2-x1)*(y2-y1))/A
            if area_frac < MIN_AREA_FRAC or area_frac > MAX_AREA_FRAC: continue
            if names is not None and cls is not None and ALLOWED_CLASSES:
                cname = names[int(cls[k])]
                if cname not in ALLOWED_CLASSES: continue
            boxes.append((x1,y1,x2,y2))
    return boxes

def boxes_to_mask(shape, boxes, inflate=INFLATE_PIXELS):
    H,W = shape[:2]; m = np.zeros((H,W), np.uint8)
    for (x1,y1,x2,y2) in boxes:
        x1i=max(0,x1-inflate); y1i=max(0,y1-inflate)
        x2i=min(W-1,x2+inflate); y2i=min(H-1,y2+inflate)
        m[y1i:y2i+1, x1i:x2i+1] = 255
    return m

def build_occupancy(mask, dilate_iter=2):
    grid = cv2.dilate(mask, np.ones((3,3), np.uint8), iterations=dilate_iter)
    return (grid>0).astype(np.uint8)

def astar(grid, start, goal):
    H,W = grid.shape
    def h(a,b): return abs(a[0]-b[0]) + abs(a[1]-b[1])
    openq=[(h(start,goal),0,start,None)]; came, cost, seen = {}, {start:0}, set()
    nbrs=[(1,0),(-1,0),(0,1),(0,-1),(1,1),(1,-1),(-1,1),(-1,-1)]
    import heapq
    while openq:
        _,g,cur,parent = heapq.heappop(openq)
        if cur in seen: continue
        seen.add(cur); came[cur]=parent
        if cur==goal: break
        for dx,dy in nbrs:
            nx,ny=cur[0]+dx, cur[1]+dy
            if not (0<=nx<H and 0<=ny<W): continue
            if grid[nx,ny]==1: continue
            ng=g+(1.4 if dx and dy else 1.0)
            if ng < cost.get((nx,ny),1e9):
                cost[(nx,ny)] = ng
                heapq.heappush(openq, (ng + h((nx,ny),goal), ng, (nx,ny), cur))
    if goal not in came: return []
    path,node=[],goal
    while node: path.append(node); node=came[node]
    return path[::-1]

def numeric_key(p):
    m = re.search(r"_img(\d+)\.jpg$", p); return int(m.group(1)) if m else 0




✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp400_avoidance.mp4
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp401_avoidance.mp4
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp402_avoidance.mp4
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp403_avoidance.mp4
skip (missing): C:\Users\ADMIN\Desktop\hospital_avoidance_demo\3_Corridor\3_Corridor\exp404
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp405_avoidance.mp4
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp406_avoidance.mp4
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp407_avoidance.mp4
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp408_avoidance.mp4
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp409_avoidance.mp4
✅ already done: C

In [4]:
def visualize_probe(frame):
    boxes = yolo_obstacles(frame, conf=0.25)
    mask  = boxes_to_mask(frame.shape, boxes, inflate=INFLATE_PIXELS)

    probe = depth_probe_mask(frame)  # force-run probe for debug
    debug = frame.copy()
    # red: YOLO mask
    red = debug[:,:,2]; red[mask>0] = 255; debug[:,:,2] = red
    # blue: probe mask
    blu = debug[:,:,0]; blu[probe>0] = 255; debug[:,:,0] = blu

    for (x1,y1,x2,y2) in boxes:
        cv2.rectangle(debug,(x1,y1),(x2,y2),(0,255,255),2)
    return debug

# test on a single frame (edit path)
test_img = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\3_Corridor\3_Corridor\exp700\exp700_img120.jpg"
frm = cv2.imread(test_img)
dbg = visualize_probe(frm)
cv2.imwrite(r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\probe_debug.jpg", dbg)
print("saved debug overlay → outputs\\probe_debug.jpg")


saved debug overlay → outputs\probe_debug.jpg


In [5]:
def overlay_masks(frame, yolo_mask, probe_mask):
    color_yolo  = np.zeros_like(frame); color_yolo[:,:,2] = 255  # red
    color_probe = np.zeros_like(frame); color_probe[:,:,0] = 255 # blue
    vis = frame.copy()
    vis = cv2.addWeighted(vis, 1.0, cv2.bitwise_and(color_yolo, color_yolo, mask=yolo_mask), 0.35, 0)
    vis = cv2.addWeighted(vis, 1.0, cv2.bitwise_and(color_probe, color_probe, mask=probe_mask), 0.35, 0)
    return vis


In [6]:
ALLOWED = {"person","chair","bench","bed","couch","tv","table","potted plant","suitcase","backpack","handbag"}  # edit as you like
MIN_AREA_FRAC = 0.002   # drop tiny specks
MAX_AREA_FRAC = 0.50    # drop absurdly big boxes
DET_CONF = 0.35         # a bit stricter

def yolo_obstacles(frame_bgr, conf=DET_CONF):
    res = yolo.predict(source=frame_bgr, conf=conf, verbose=False, device=0 if device=="cuda" else None)[0]
    H, W = frame_bgr.shape[:2]
    A = H*W
    boxes = []
    if res.boxes is not None:
        xyxy = res.boxes.xyxy.cpu().numpy().astype(int)
        cls  = res.boxes.cls.cpu().numpy().astype(int)
        names = yolo.model.names
        for (x1,y1,x2,y2), c in zip(xyxy, cls):
            w = max(1, x2-x1); h = max(1, y2-y1)
            area_frac = (w*h)/A
            if names[c] not in ALLOWED:    continue
            if area_frac < MIN_AREA_FRAC:  continue
            if area_frac > MAX_AREA_FRAC:  continue
            boxes.append((x1,y1,x2,y2))
    return boxes


In [7]:
USE_DEPTH_PROBE  = True
PROBE_EVERY      = 6
PROBE_WIDTH_FRAC = 0.30
PROBE_TARGET_W   = 400   # a tad larger for clarity

def depth_probe_mask(frame_bgr):
    _ensure_depth_model()
    H, W = frame_bgr.shape[:2]
    cx1 = int(W*(0.5 - PROBE_WIDTH_FRAC/2)); cx2 = int(W*(0.5 + PROBE_WIDTH_FRAC/2))
    roi = frame_bgr[:, cx1:cx2]
    h, w = roi.shape[:2]
    tw = PROBE_TARGET_W; th = int(h * (tw / max(1, w)))
    roi_small = cv2.resize(roi, (tw, th))
    rgb = cv2.cvtColor(roi_small, cv2.COLOR_BGR2RGB)

    with torch.no_grad():
        inputs = depth_proc(images=rgb, return_tensors="pt").to(device)
        depth = depth_model(**inputs).predicted_depth.squeeze().detach().cpu().numpy()

    d = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)
    d8 = 255 - (d*255).astype(np.uint8)   # nearer = bright (works well with DPT)
    # Auto threshold: Otsu (fallback to percentile if needed)
    th, mask_small = cv2.threshold(d8, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    # If Otsu too tight (rare), use percentile:
    if mask_small.mean() < 5:  # all black-ish
        th = np.percentile(d8, 80)  # top 20% as near
        _, mask_small = cv2.threshold(d8, th, 255, cv2.THRESH_BINARY)

    mask = cv2.resize(mask_small, (cx2-cx1, H), interpolation=cv2.INTER_NEAREST)
    full = np.zeros((H, W), dtype=np.uint8)
    full[:, cx1:cx2] = mask
    full = cv2.morphologyEx(full, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8), iterations=1)
    full = cv2.dilate(full, np.ones((3,3), np.uint8), iterations=1)
    return full


In [8]:
def visualize_probe(frame):
    boxes = yolo_obstacles(frame, conf=DET_CONF)
    ymask = boxes_to_mask(frame.shape, boxes, inflate=INFLATE_PIXELS)
    pmask = depth_probe_mask(frame)
    vis   = overlay_masks(frame, ymask, pmask)

    # coverage stats
    H,W = frame.shape[:2]
    ycov = (ymask>0).sum()/(H*W)
    pcov = (pmask>0).sum()/(H*W)
    cv2.putText(vis, f"YOLO mask {ycov*100:.1f}% | Probe {pcov*100:.1f}%", (10,30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
    for (x1,y1,x2,y2) in boxes:
        cv2.rectangle(vis,(x1,y1),(x2,y2),(0,255,255),2)
    return vis


In [9]:
test_img = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\3_Corridor\3_Corridor\exp700\exp700_img120.jpg"
frm = cv2.imread(test_img)
dbg = visualize_probe(frm)
cv2.imwrite(r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\probe_debug.jpg", dbg)
print("saved → outputs\\probe_debug.jpg")


saved → outputs\probe_debug.jpg


✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp400_avoidance.mp4
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp401_avoidance.mp4
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp402_avoidance.mp4
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp403_avoidance.mp4
skip (missing): C:\Users\ADMIN\Desktop\hospital_avoidance_demo\3_Corridor\3_Corridor\exp404
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp405_avoidance.mp4
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp406_avoidance.mp4
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp407_avoidance.mp4
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp408_avoidance.mp4
✅ already done: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp409_avoidance.mp4
✅ already done: C

In [14]:
import cv2, os

VID_DIR = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos"
demo_out = os.path.join(VID_DIR, "hospital_demo.mp4")
picks = [400, 685, 700, 717, 728, 731]  # <- change this list to your favorites

# find properties from the first valid file
cap0 = None
for e in picks:
    p = os.path.join(VID_DIR, f"exp{e}_avoidance.mp4")
    if os.path.exists(p):
        cap0 = cv2.VideoCapture(p)
        if cap0.isOpened(): break
if cap0 is None or not cap0.isOpened():
    raise RuntimeError("No input videos found in picks.")

fps = cap0.get(cv2.CAP_PROP_FPS) or 20.0
W   = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
H   = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap0.release()

writer = cv2.VideoWriter(demo_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W,H))

for e in picks:
    path = os.path.join(VID_DIR, f"exp{e}_avoidance.mp4")
    if not os.path.exists(path):
        print("skip (missing clip):", path); 
        continue
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        print("skip (bad clip):", path); 
        continue

    # optional: write a quick title card for each segment
    title = f"exp{e}: YOLO + Adaptive Depth Probe → Occupancy → A*"
    tc = (np.zeros((H, W, 3), np.uint8) + 32)
    cv2.putText(tc, title, (30, int(H*0.55)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
    for _ in range(int(fps*0.8)):  # ~0.8s title
        writer.write(tc)

    while True:
        ret, fr = cap.read()
        if not ret: break
        writer.write(fr)
    cap.release()

writer.release()
print("🎬 merged demo →", demo_out)


🎬 merged demo → C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\hospital_demo.mp4


In [15]:
SECONDS_PER_CLIP = 6   # trim each to 6s

# ... in the merge loop, replace the while with:
max_frames = int(fps * SECONDS_PER_CLIP)
fcount = 0
while fcount < max_frames:
    ret, fr = cap.read()
    if not ret: break
    writer.write(fr)
    fcount += 1


In [16]:
# --- reprocess exp400 & exp401 with more sensitive settings ---
import os, cv2, numpy as np, glob, torch, heapq, re
from ultralytics import YOLO
from transformers import DPTForDepthEstimation, DPTImageProcessor

BASE = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\3_Corridor\3_Corridor"
OUT  = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos"
os.makedirs(OUT, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
yolo = YOLO("yolov8n.pt")

# --- overrides just for these two exps ---
DET_CONF       = 0.32          # more sensitive (was 0.45)
MIN_AREA_FRAC  = 0.0015        # allow smaller boxes
MAX_AREA_FRAC  = 0.55          # allow larger boxes early
INFLATE_PIXELS = 10            # slightly wider bubble
REPLAN_EVERY   = 4             # replan a bit more often

# depth probe: wider, more frequent, slightly lower percentile
USE_DEPTH_PROBE  = True
PROBE_EVERY      = 3
PROBE_WIDTH_FRAC = 0.34
PROBE_TARGET_W   = 448

# adaptive depth knobs
ADAPT_PERCENTILE = 75          # was 80; more sensitive
DILATE_ITER      = 3

# --- load MiDaS once ---
depth_model = DPTForDepthEstimation.from_pretrained("Intel/dpt-hybrid-midas").to(device)
depth_proc  = DPTImageProcessor.from_pretrained("Intel/dpt-hybrid-midas")

def preprocess(img):
    # mild gamma + CLAHE for low contrast / overexposed frames
    g = 0.9
    img_f = np.clip((img/255.0)**g, 0, 1.0)
    img_u = (img_f*255).astype(np.uint8)
    lab = cv2.cvtColor(img_u, cv2.COLOR_BGR2LAB)
    l,a,b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2BGR)

def yolo_obstacles(frame_bgr):
    res = yolo.predict(source=frame_bgr, conf=DET_CONF, verbose=False,
                       device=0 if device=="cuda" else None)[0]
    H,W = frame_bgr.shape[:2]; A = H*W
    boxes=[]
    if res.boxes is not None:
        xyxy = res.boxes.xyxy.cpu().numpy().astype(int)
        for (x1,y1,x2,y2) in xyxy:
            area = max(1,(x2-x1)*(y2-y1))/A
            if area < MIN_AREA_FRAC or area > MAX_AREA_FRAC: continue
            boxes.append((x1,y1,x2,y2))
    return boxes

def boxes_to_mask(shape, boxes, inflate=INFLATE_PIXELS):
    H,W = shape[:2]; m = np.zeros((H,W), np.uint8)
    for (x1,y1,x2,y2) in boxes:
        x1i=max(0,x1-inflate); y1i=max(0,y1-inflate)
        x2i=min(W-1,x2+inflate); y2i=min(H-1,y2+inflate)
        m[y1i:y2i+1, x1i:x2i+1] = 255
    return m

def depth_probe_mask(frame_bgr):
    H,W = frame_bgr.shape[:2]
    cx1 = int(W*(0.5-PROBE_WIDTH_FRAC/2)); cx2 = int(W*(0.5+PROBE_WIDTH_FRAC/2))
    roi = frame_bgr[:, cx1:cx2]
    h,w = roi.shape[:2]; tw = PROBE_TARGET_W; th = int(h*(tw/max(1,w)))
    roi_small = cv2.resize(roi,(tw,th))
    rgb = cv2.cvtColor(roi_small, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        inputs = depth_proc(images=rgb, return_tensors="pt").to(device)
        d = depth_model(**inputs).predicted_depth.squeeze().detach().cpu().numpy()
    d = (d - d.min())/(d.max()-d.min()+1e-8)
    d8 = 255 - (d*255).astype(np.uint8)
    d8 = cv2.resize(d8, (cx2-cx1, H), interpolation=cv2.INTER_LINEAR)
    thr = np.percentile(d8, ADAPT_PERCENTILE)
    mask = (d8 >= thr).astype(np.uint8)*255
    full = np.zeros((H,W), np.uint8); full[:, cx1:cx2] = mask
    full = cv2.dilate(full, np.ones((3,3),np.uint8), iterations=DILATE_ITER)
    return full

def build_occupancy(mask, dilate_iter=2):
    grid = cv2.dilate(mask, np.ones((3,3),np.uint8), iterations=dilate_iter)
    return (grid>0).astype(np.uint8)

def astar(grid, start, goal):
    H,W = grid.shape
    def h(a,b): return abs(a[0]-b[0])+abs(a[1]-b[1])
    openq=[(h(start,goal),0,start,None)]; came,cost,seen={}, {start:0}, set()
    nbrs=[(1,0),(-1,0),(0,1),(0,-1),(1,1),(1,-1),(-1,1),(-1,-1)]
    while openq:
        _,g,cur,parent = heapq.heappop(openq)
        if cur in seen: continue
        seen.add(cur); came[cur]=parent
        if cur==goal: break
        for dx,dy in nbrs:
            nx,ny = cur[0]+dx, cur[1]+dy
            if not (0<=nx<H and 0<=ny<W): continue
            if grid[nx,ny]==1: continue
            ng = g + (1.4 if dx and dy else 1.0)
            if ng < cost.get((nx,ny), 1e9):
                cost[(nx,ny)] = ng; heapq.heappush(openq,(ng+h((nx,ny),goal),ng,(nx,ny),cur))
    if goal not in came: return []
    path,node=[],goal
    while node: path.append(node); node=came[node]
    return path[::-1]

def numeric_key(p):
    m=re.search(r"_img(\d+)\.jpg$",p); return int(m.group(1)) if m else 0

for exp in [400, 401]:  # only first two
    exp_dir = rf"{BASE}\exp{exp}"
    out_path = rf"{OUT}\exp{exp}_avoidance.mp4"
    # force overwrite
    if os.path.exists(out_path): os.remove(out_path)

    frames = sorted(glob.glob(rf"{exp_dir}\exp{exp}_img*.jpg"), key=numeric_key)
    if not frames: 
        print("no frames:", exp_dir); 
        continue

    first = cv2.imread(frames[0]); H,W = first.shape[:2]
    vw = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), 20.0, (W,H))
    print(f"reprocessing exp{exp}: {len(frames)} frames")

    last_path=[]
    for i,fp in enumerate(frames,1):
        raw = cv2.imread(fp)
        if raw is None: continue
        frame = preprocess(raw)

        boxes = yolo_obstacles(frame)
        mask  = boxes_to_mask(frame.shape, boxes)

        if USE_DEPTH_PROBE and (i % PROBE_EVERY == 0):
            mask = cv2.bitwise_or(mask, depth_probe_mask(frame))

        grid  = build_occupancy(mask, dilate_iter=2)
        start = (int(H*0.85), int(W*0.5))
        goal  = (int(H*0.15), int(W*0.5))
        if not last_path or (i % REPLAN_EVERY == 0):
            last_path = astar(grid, start, goal)

        vis = raw.copy()
        for (x1,y1,x2,y2) in boxes:
            cv2.rectangle(vis,(x1,y1),(x2,y2),(0,0,255),2)
        for (r,c) in last_path:
            cv2.circle(vis,(c,r),2,(0,255,0),-1)
        cv2.putText(vis,f"exp{exp} {i}/{len(frames)} boxes:{len(boxes)} path:{len(last_path)}",
                    (10,25), cv2.FONT_HERSHEY_SIMPLEX, 0.7,(255,255,255),2)
        vw.write(vis)

    vw.release()
    print("✅ overwritten →", out_path)


reprocessing exp400: 229 frames
✅ overwritten → C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp400_avoidance.mp4
reprocessing exp401: 170 frames
✅ overwritten → C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp401_avoidance.mp4


In [17]:
import cv2, os, numpy as np

VID_DIR  = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos"
demo_out = os.path.join(VID_DIR, "hospital_demo.mp4")

# pick your favorites (include 400/401 to show the fix)
picks = [400, 401, 685, 700, 717, 728, 731]

# discover size/fps from first valid
cap0 = None
for e in picks:
    p = os.path.join(VID_DIR, f"exp{e}_avoidance.mp4")
    if os.path.exists(p):
        cap0 = cv2.VideoCapture(p)
        if cap0.isOpened(): break
assert cap0 and cap0.isOpened(), "No valid input videos"
fps = cap0.get(cv2.CAP_PROP_FPS) or 20.0
W   = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
H   = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap0.release()

writer = cv2.VideoWriter(demo_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W,H))

SECONDS_PER_CLIP = 6  # trim each clip to keep the demo snappy

for e in picks:
    path = os.path.join(VID_DIR, f"exp{e}_avoidance.mp4")
    if not os.path.exists(path):
        print("skip (missing):", path); continue

    # quick title card per segment
    tc = (np.zeros((H, W, 3), np.uint8) + 32)
    title = f"exp{e}: YOLO + Adaptive Depth → Occupancy → A*"
    cv2.putText(tc, title, (30, int(H*0.55)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
    for _ in range(int(fps*0.8)): writer.write(tc)

    cap = cv2.VideoCapture(path)
    max_frames = int(fps * SECONDS_PER_CLIP)
    fcount = 0
    while fcount < max_frames:
        ret, fr = cap.read()
        if not ret: break
        writer.write(fr); fcount += 1
    cap.release()

writer.release()
print("🎬 merged demo →", demo_out)


🎬 merged demo → C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\hospital_demo.mp4


In [18]:
# --- first-N-frames overrides (more aggressive just at the start) ---
AGG_FRAMES          = 60           # ~3 sec @ 20 FPS
AGG_DET_CONF        = 0.28         # more sensitive than 0.32
AGG_MIN_AREA_FRAC   = 0.0010
AGG_MAX_AREA_FRAC   = 0.60
AGG_INFLATE_PIXELS  = 12
AGG_PROBE_EVERY     = 1
AGG_PROBE_WIDTH     = 0.38
AGG_ADAPT_PERCENT   = 70           # lower percentile -> more near pixels


In [19]:
def edge_plane_mask(frame_bgr, width_frac=0.34, canny1=50, canny2=120, v_sobel_thresh=60):
    """
    Backup mask using vertical edges in center ROI.
    Marks a vertical band as blocked if a strong vertical edge plane (door/wall) is present.
    """
    H, W = frame_bgr.shape[:2]
    cx1 = int(W*(0.5 - width_frac/2))
    cx2 = int(W*(0.5 + width_frac/2))
    roi = frame_bgr[:, cx1:cx2]

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    # light denoise for blur cases
    gray = cv2.GaussianBlur(gray, (5,5), 0)

    # edges and vertical gradient
    edges = cv2.Canny(gray, canny1, canny2)
    sobelx = cv2.Sobel(gray, cv2.CV_16S, 1, 0, ksize=3)
    sobelx = cv2.convertScaleAbs(sobelx)

    # combine: edges OR strong vertical gradient
    m1 = edges
    _, m2 = cv2.threshold(sobelx, v_sobel_thresh, 255, cv2.THRESH_BINARY)
    m  = cv2.bitwise_or(m1, m2)

    # thicken & fill small gaps
    m = cv2.dilate(m, np.ones((5,5), np.uint8), iterations=1)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((7,7), np.uint8), iterations=1)

    # build full-width mask
    full = np.zeros((H, W), dtype=np.uint8)
    full[:, cx1:cx2] = m
    # optional: only keep lower 80% (ignore ceiling lines)
    keep_from = int(H*0.20)
    full[:keep_from, :] = 0
    return full


In [20]:
last_path = []
for i, fp in enumerate(frames, 1):
    raw = cv2.imread(fp)
    if raw is None: 
        continue

    frame = preprocess(raw)

    # ---- choose aggressive vs normal for first N frames ----
    early = (i <= AGG_FRAMES)
    det_conf       = AGG_DET_CONF       if early else DET_CONF
    min_area_frac  = AGG_MIN_AREA_FRAC  if early else MIN_AREA_FRAC
    max_area_frac  = AGG_MAX_AREA_FRAC  if early else MAX_AREA_FRAC
    inflate_px     = AGG_INFLATE_PIXELS if early else INFLATE_PIXELS
    probe_every    = AGG_PROBE_EVERY    if early else PROBE_EVERY
    probe_width    = AGG_PROBE_WIDTH    if early else PROBE_WIDTH_FRAC
    adapt_percent  = AGG_ADAPT_PERCENT  if early else ADAPT_PERCENTILE

    # ---- YOLO with per-frame thresholds ----
    # (copy of yolo_obstacles with dynamic thresholds)
    res = yolo.predict(source=frame, conf=det_conf, verbose=False,
                       device=0 if device=="cuda" else None)[0]
    H, W = frame.shape[:2]; A = H*W
    boxes=[]
    if res.boxes is not None:
        xyxy = res.boxes.xyxy.cpu().numpy().astype(int)
        for (x1,y1,x2,y2) in xyxy:
            # optional: ignore boxes starting too high (floor relevance)
            if y1 < int(0.25*H): 
                continue
            area = max(1,(x2-x1)*(y2-y1))/A
            if area < min_area_frac or area > max_area_frac: 
                continue
            boxes.append((x1,y1,x2,y2))

    # ---- YOLO mask ----
    mask = boxes_to_mask(frame.shape, boxes, inflate=inflate_px)

    # ---- depth probe (adaptive percentile) ----
    if USE_DEPTH_PROBE and (i % probe_every == 0):
        # temporary probe using current width/percentile
        Hf, Wf = frame.shape[:2]
        cx1 = int(Wf*(0.5 - probe_width/2)); cx2 = int(Wf*(0.5 + probe_width/2))
        roi = frame[:, cx1:cx2]
        h,w = roi.shape[:2]; tw = PROBE_TARGET_W; th = int(h*(tw/max(1,w)))
        roi_small = cv2.resize(roi,(tw,th))
        rgb = cv2.cvtColor(roi_small, cv2.COLOR_BGR2RGB)
        with torch.no_grad():
            inputs = depth_proc(images=rgb, return_tensors="pt").to(device)
            d = depth_model(**inputs).predicted_depth.squeeze().detach().cpu().numpy()
        d = (d - d.min())/(d.max()-d.min()+1e-8)
        d8 = 255 - (d*255).astype(np.uint8)
        d8 = cv2.resize(d8, (cx2-cx1, Hf), interpolation=cv2.INTER_LINEAR)
        thr = np.percentile(d8, adapt_percent)
        probe = (d8 >= thr).astype(np.uint8)*255
        probe_full = np.zeros((Hf,Wf), np.uint8); probe_full[:, cx1:cx2] = probe
        probe_full = cv2.dilate(probe_full, np.ones((3,3),np.uint8), iterations=DILATE_ITER)
        mask = cv2.bitwise_or(mask, probe_full)

    # ---- add edge-based door plane (backup) ----
    edge_mask = edge_plane_mask(frame, width_frac=probe_width)
    mask = cv2.bitwise_or(mask, edge_mask)

    # ---- occupancy + plan ----
    grid  = build_occupancy(mask, dilate_iter=2)
    H, W = grid.shape
    start = (int(H*0.85), int(W*0.5))
    goal  = (int(H*0.15), int(W*0.5))
    need_replan = (not last_path) or (i % REPLAN_EVERY == 0)
    if need_replan:
        last_path = astar(grid, start, goal)

    # ---- Fallback: if path cuts through blocked center band, recompute aggressively ----
    if last_path:
        # check a narrow band around the center columns in the *lower* quarter
        band = grid[int(H*0.70):, int(W*0.45):int(W*0.55)]
        too_open = (band.mean() < 0.04)  # almost free -> suspicious if door is actually there
        # Or: detect intersection directly
        path_hits = any(grid[r, c] == 1 for (r,c) in last_path[:max(10, len(last_path)//4)])
        if too_open or path_hits:
            # widen probe + inflate and re-plan once
            edge_mask2 = edge_plane_mask(frame, width_frac=min(0.42, probe_width+0.06))
            mask2 = cv2.bitwise_or(mask, edge_mask2)
            mask2 = cv2.dilate(mask2, np.ones((5,5),np.uint8), iterations=1)
            grid2 = build_occupancy(mask2, dilate_iter=2)
            last_path = astar(grid2, start, goal)
            grid = grid2  # keep for STOP check & overlay

    # ---- optional STOP if blocked across bottom center ----
    vis = raw.copy()
    bottom = grid[int(H*0.80):, int(W*0.40):int(W*0.60)]
    blocked = (bottom.mean() > 0.18)
    if blocked:
        last_path = []
        cv2.putText(vis, "STOP: BLOCKED AHEAD", (10, H-20), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

    # ---- draw ----
    for (x1,y1,x2,y2) in boxes:
        cv2.rectangle(vis,(x1,y1),(x2,y2),(0,0,255),2)
    for (r,c) in last_path:
        cv2.circle(vis,(c,r),2,(0,255,0),-1)

    cv2.putText(vis, f"{i}/{len(frames)} boxes:{len(boxes)} path:{len(last_path)}{' AGG' if early else ''}",
                (10,25), cv2.FONT_HERSHEY_SIMPLEX, 0.7,(255,255,255),2)
    vw.write(vis)



In [21]:
for exp in [400, 401]:
    exp_dir  = rf"{BASE}\exp{exp}"
    out_path = rf"{OUT}\exp{exp}_avoidance.mp4"
    if os.path.exists(out_path): os.remove(out_path)  # force overwrite

    frames = sorted(glob.glob(rf"{exp_dir}\exp{exp}_img*.jpg"), key=numeric_key)
    if not frames: 
        print("no frames:", exp_dir); 
        continue

    first = cv2.imread(frames[0]); H, W = first.shape[:2]
    vw = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), 20.0, (W, H))
    print(f"reprocessing exp{exp}: {len(frames)} frames")

    # ⬇️ paste your inner loop here exactly as you have it ⬇️

    vw.release()
    print("✅ overwritten →", out_path)


reprocessing exp400: 229 frames
✅ overwritten → C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp400_avoidance.mp4
reprocessing exp401: 170 frames
✅ overwritten → C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp401_avoidance.mp4


In [23]:
DEBUG_DIR = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\debug_exp400"
os.makedirs(DEBUG_DIR, exist_ok=True)

if exp == 400 and i <= 60:
    # save masks and vis
    cv2.imwrite(os.path.join(DEBUG_DIR, f"frame_{i:03d}_vis.jpg"), vis)
    cv2.imwrite(os.path.join(DEBUG_DIR, f"frame_{i:03d}_mask.png"), mask)
    cv2.imwrite(os.path.join(DEBUG_DIR, f"frame_{i:03d}_grid.png"), (grid*255).astype(np.uint8))
-------------------------------------------------------------------------

In [37]:
# --- imports ---
import os, re, glob, time
import cv2, numpy as np
import torch, heapq
from ultralytics import YOLO
from transformers import DPTForDepthEstimation, DPTImageProcessor


In [38]:
# --- paths ---
BASE = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\3_Corridor\3_Corridor"
OUT  = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos"
os.makedirs(OUT, exist_ok=True)

# (optional) debug folder
DEBUG_DIR = os.path.join(OUT, "debug_exp400")
os.makedirs(DEBUG_DIR, exist_ok=True)


In [39]:
# --- device & models ---
device = "cuda" if torch.cuda.is_available() else "cpu"
yolo = YOLO("yolov8n.pt")  # swap to fine-tuned weights if you have them

# depth (MiDaS)
depth_model = DPTForDepthEstimation.from_pretrained("Intel/dpt-hybrid-midas").to(device)
depth_proc  = DPTImageProcessor.from_pretrained("Intel/dpt-hybrid-midas")


In [40]:
# --- normal tuning ---
DET_CONF        = 0.32
MIN_AREA_FRAC   = 0.0015
MAX_AREA_FRAC   = 0.55
INFLATE_PIXELS  = 14     # updated from 10 -> 14
REPLAN_EVERY    = 4

# --- depth probe (adaptive percentile) ---
USE_DEPTH_PROBE  = True
PROBE_EVERY      = 1      # updated from 3 -> 1 (probe every frame)
PROBE_WIDTH_FRAC = 0.40   # updated from 0.34 -> 0.40 (wider center strip)
PROBE_TARGET_W   = 448
ADAPT_PERCENTILE = 65     # updated from 75 -> 65 (more sensitive)
DILATE_ITER      = 3      # updated/confirmed

# --- stop rule threshold ---
STOP_BLOCKED_THRESH = 0.14  # trigger STOP if bottom-center occupancy > this

# --- first-N-frames aggressive overrides (unchanged unless you want them adjusted) ---
AGG_FRAMES          = 60
AGG_DET_CONF        = 0.28
AGG_MIN_AREA_FRAC   = 0.0010
AGG_MAX_AREA_FRAC   = 0.60
AGG_INFLATE_PIXELS  = 12
AGG_PROBE_EVERY     = 1
AGG_PROBE_WIDTH     = 0.38
AGG_ADAPT_PERCENT   = 70


In [41]:
def preprocess(img):
    g = 0.9
    img_f = np.clip((img/255.0)**g, 0, 1.0)
    img_u = (img_f*255).astype(np.uint8)
    lab = cv2.cvtColor(img_u, cv2.COLOR_BGR2LAB)
    l,a,b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2BGR)


In [48]:
def yolo_obstacles_dynamic(frame_bgr, det_conf, min_area_frac, max_area_frac):
    res = yolo.predict(source=frame_bgr, conf=det_conf, verbose=False,
                       device=0 if device=="cuda" else None)[0]
    H, W = frame_bgr.shape[:2]; A = H*W
    boxes=[]
    if res.boxes is not None:
        xyxy = res.boxes.xyxy.cpu().numpy().astype(int)
        for (x1,y1,x2,y2) in xyxy:
            # ignore very high boxes (ceiling clutter)
            if y1 < int(0.25*H): 
                continue
            area = max(1,(x2-x1)*(y2-y1))/A
            if area < min_area_frac or area > max_area_frac: 
                continue
            boxes.append((x1,y1,x2,y2))
    return boxes

# --- edge-plane fallback: catches flat white walls/cabinets in center band ---
def edge_plane_mask(frame_bgr,
                    width_frac=0.40,        # use same as PROBE_WIDTH_FRAC or set explicitly
                    edge_min_density=0.02,  # if edges < 2% → likely flat surface
                    bright_min=110,         # mean gray ≥ this → bright wall/cabinet
                    dilate_iter=3):
    """
    Returns a binary mask (H×W, 255=obstacle) marking a center vertical band as
    obstacle when it's low-texture & bright (typical white wall/cabinet).
    """
    H, W = frame_bgr.shape[:2]
    cx1 = int(W * (0.5 - width_frac / 2.0))
    cx2 = int(W * (0.5 + width_frac / 2.0))
    roi = frame_bgr[:, cx1:cx2]

    # enhance a bit, measure texture and brightness
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    edges = cv2.Canny(gray, 40, 120)
    edge_density = (edges > 0).mean()                  # 0..1
    mean_bright  = gray.mean()                         # 0..255

    # flat + bright → treat as wall/cabinet
    if edge_density < edge_min_density and mean_bright >= bright_min:
        m = np.zeros((H, W), np.uint8)
        m[:, cx1:cx2] = 255
        m = cv2.dilate(m, np.ones((3, 3), np.uint8), iterations=dilate_iter)
        return m
    else:
        return np.zeros((H, W), np.uint8)


def boxes_to_mask(shape, boxes, inflate):
    H,W = shape[:2]; m = np.zeros((H,W), np.uint8)
    for (x1,y1,x2,y2) in boxes:
        x1i=max(0,x1-inflate); y1i=max(0,y1-inflate)
        x2i=min(W-1,x2+inflate); y2i=min(H-1,y2+inflate)
        m[y1i:y2i+1, x1i:x2i+1] = 255
    return m

def build_occupancy(mask, dilate_iter=2):
    grid = cv2.dilate(mask, np.ones((3,3), np.uint8), iterations=dilate_iter)
    return (grid>0).astype(np.uint8)

def astar(grid, start, goal):
    H,W = grid.shape
    def h(a,b): return abs(a[0]-b[0]) + abs(a[1]-b[1])
    openq=[(h(start,goal),0,start,None)]
    came, cost, seen = {}, {start:0}, set()
    nbrs=[(1,0),(-1,0),(0,1),(0,-1),(1,1),(1,-1),(-1,1),(-1,-1)]
    while openq:
        _,g,cur,parent = heapq.heappop(openq)
        if cur in seen: continue
        seen.add(cur); came[cur]=parent
        if cur==goal: break
        for dx,dy in nbrs:
            nx,ny = cur[0]+dx, cur[1]+dy
            if not (0<=nx<H and 0<=ny<W): continue
            if grid[nx,ny]==1: continue
            ng = g + (1.4 if dx and dy else 1.0)
            if ng < cost.get((nx,ny),1e9):
                cost[(nx,ny)] = ng
                heapq.heappush(openq, (ng + h((nx,ny),goal), ng, (nx,ny), cur))
    if goal not in came: return []
    path,node=[],goal
    while node: path.append(node); node=came[node]
    return path[::-1]

def numeric_key(p):
    m = re.search(r"_img(\d+)\.jpg$", p)
    return int(m.group(1)) if m else 0


In [43]:
def depth_probe_adaptive(frame_bgr, probe_width, adapt_percent):
    Hf, Wf = frame_bgr.shape[:2]
    cx1 = int(Wf*(0.5 - probe_width/2)); cx2 = int(Wf*(0.5 + probe_width/2))
    roi = frame_bgr[:, cx1:cx2]
    h,w = roi.shape[:2]; tw = PROBE_TARGET_W; th = int(h*(tw/max(1,w)))
    roi_small = cv2.resize(roi,(tw,th))
    rgb = cv2.cvtColor(roi_small, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        inputs = depth_proc(images=rgb, return_tensors="pt").to(device)
        d = depth_model(**inputs).predicted_depth.squeeze().detach().cpu().numpy()
    d = (d - d.min())/(d.max()-d.min()+1e-8)
    d8 = 255 - (d*255).astype(np.uint8)
    d8 = cv2.resize(d8, (cx2-cx1, Hf), interpolation=cv2.INTER_LINEAR)
    thr = np.percentile(d8, adapt_percent)
    probe = (d8 >= thr).astype(np.uint8)*255
    probe_full = np.zeros((Hf,Wf), np.uint8); probe_full[:, cx1:cx2] = probe
    probe_full = cv2.dilate(probe_full, np.ones((3,3),np.uint8), iterations=DILATE_ITER)
    return probe_full

def edge_plane_mask(frame_bgr, width_frac=0.34, canny1=50, canny2=120, v_sobel_thresh=60):
    H, W = frame_bgr.shape[:2]
    cx1 = int(W*(0.5 - width_frac/2))
    cx2 = int(W*(0.5 + width_frac/2))
    roi = frame_bgr[:, cx1:cx2]
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5,5), 0)
    edges = cv2.Canny(gray, canny1, canny2)
    sobelx = cv2.Sobel(gray, cv2.CV_16S, 1, 0, ksize=3)
    sobelx = cv2.convertScaleAbs(sobelx)
    _, m2 = cv2.threshold(sobelx, v_sobel_thresh, 255, cv2.THRESH_BINARY)
    m  = cv2.bitwise_or(edges, m2)
    m  = cv2.dilate(m, np.ones((5,5), np.uint8), iterations=1)
    m  = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((7,7), np.uint8), iterations=1)
    full = np.zeros((H, W), dtype=np.uint8); full[:, cx1:cx2] = m
    full[:int(H*0.20), :] = 0
    return full


In [44]:
for exp in [400, 401]:
    exp_dir  = rf"{BASE}\exp{exp}"
    out_path = rf"{OUT}\exp{exp}_avoidance.mp4"
    if not os.path.isdir(exp_dir):
        print("missing:", exp_dir); 
        continue

    # force overwrite (we’re fixing these)
    if os.path.exists(out_path): os.remove(out_path)

    frames = sorted(glob.glob(rf"{exp_dir}\exp{exp}_img*.jpg"), key=numeric_key)
    if not frames:
        print("no frames:", exp_dir); 
        continue

    first = cv2.imread(frames[0]); H, W = first.shape[:2]
    vw = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), 20.0, (W, H))
    print(f"reprocessing exp{exp}: {len(frames)} frames")

    last_path=[]
    for i, fp in enumerate(frames, 1):
        raw = cv2.imread(fp)
        if raw is None: 
            continue

        frame = preprocess(raw)

        # choose aggressive vs normal
        early = (i <= AGG_FRAMES)
        det_conf       = AGG_DET_CONF       if early else DET_CONF
        min_area_frac  = AGG_MIN_AREA_FRAC  if early else MIN_AREA_FRAC
        max_area_frac  = AGG_MAX_AREA_FRAC  if early else MAX_AREA_FRAC
        inflate_px     = AGG_INFLATE_PIXELS if early else INFLATE_PIXELS
        probe_every    = AGG_PROBE_EVERY    if early else PROBE_EVERY
        probe_width    = AGG_PROBE_WIDTH    if early else PROBE_WIDTH_FRAC
        adapt_percent  = AGG_ADAPT_PERCENT  if early else ADAPT_PERCENTILE

        # YOLO with dynamic thresholds
        boxes = yolo_obstacles_dynamic(frame, det_conf, min_area_frac, max_area_frac)
        mask  = boxes_to_mask(frame.shape, boxes, inflate=inflate_px)

        # adaptive depth probe
        if USE_DEPTH_PROBE and (i % probe_every == 0):
            mask = cv2.bitwise_or(mask, depth_probe_adaptive(frame, probe_width, adapt_percent))

        # edge-based backup
        mask = cv2.bitwise_or(mask, edge_plane_mask(frame, width_frac=probe_width))

        # occupancy + plan
        grid  = build_occupancy(mask, dilate_iter=2)
        start = (int(H*0.85), int(W*0.5))
        goal  = (int(H*0.15), int(W*0.5))
        if (not last_path) or (i % REPLAN_EVERY == 0):
            last_path = astar(grid, start, goal)

        # fallback: if path intersects blocked center, recompute once wider
        if last_path:
            band = grid[int(H*0.70):, int(W*0.45):int(W*0.55)]
            too_open  = (band.mean() < 0.04)
            path_hits = any(grid[r, c] == 1 for (r,c) in last_path[:max(10, len(last_path)//4)])
            if too_open or path_hits:
                mask2 = cv2.bitwise_or(mask, edge_plane_mask(frame, width_frac=min(0.42, probe_width+0.06)))
                mask2 = cv2.dilate(mask2, np.ones((5,5),np.uint8), iterations=1)
                grid2 = build_occupancy(mask2, dilate_iter=2)
                last_path = astar(grid2, start, goal)
                grid = grid2

        # optional STOP banner if bottom center is blocked
        vis = raw.copy()
        bottom = grid[int(H*0.80):, int(W*0.40):int(W*0.60)]
        if (bottom.mean() > 0.18):
            last_path = []
            cv2.putText(vis, "STOP: BLOCKED AHEAD", (10, H-20), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

        # draw & write
        for (x1,y1,x2,y2) in boxes:
            cv2.rectangle(vis,(x1,y1),(x2,y2),(0,0,255),2)
        for (r,c) in last_path:
            cv2.circle(vis,(c,r),2,(0,255,0),-1)

        cv2.putText(vis, f"exp{exp} {i}/{len(frames)} boxes:{len(boxes)} path:{len(last_path)}{' AGG' if early else ''}",
                    (10,25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        vw.write(vis)

    vw.release()
    print("✅ overwritten →", out_path)


reprocessing exp400: 229 frames
✅ overwritten → C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp400_avoidance.mp4
reprocessing exp401: 170 frames
✅ overwritten → C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp401_avoidance.mp4


In [47]:
import os, cv2, numpy as np

VID_DIR  = OUT
demo_out = os.path.join(VID_DIR, "hospital_demo1.mp4")
picks = [400, 401, 700, 717, 728, 731]  # change as you like

# probe size/fps
cap0=None
for e in picks:
    p = os.path.join(VID_DIR, f"exp{e}_avoidance.mp4")
    if os.path.exists(p):
        cap0 = cv2.VideoCapture(p)
        if cap0.isOpened(): break
assert cap0 and cap0.isOpened(), "No valid input videos"
fps = cap0.get(cv2.CAP_PROP_FPS) or 20.0
W   = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
H   = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap0.release()

writer = cv2.VideoWriter(demo_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
SECONDS_PER_CLIP = 6

for e in picks:
    path = os.path.join(VID_DIR, f"exp{e}_avoidance.mp4")
    if not os.path.exists(path): 
        print("skip (missing):", path); 
        continue

    # title card
    tc = (np.zeros((H, W, 3), np.uint8) + 32)
    title = f"exp{e}: YOLO + Adaptive Depth → Occupancy → A*"
    cv2.putText(tc, title, (30, int(H*0.55)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
    for _ in range(int(fps*0.8)): writer.write(tc)

    cap = cv2.VideoCapture(path)
    max_frames = int(fps * SECONDS_PER_CLIP)
    fcount = 0
    while fcount < max_frames:
        ret, fr = cap.read()
        if not ret: break
        writer.write(fr); fcount += 1
    cap.release()

writer.release()
print("🎬 merged demo →", demo_out)


🎬 merged demo → C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\hospital_demo1.mp4


In [ ]:
# ===== MAIN LOOP (updated) =====
import os, glob, cv2, numpy as np

# --- extra settings ---
STOP_BLOCKED_THRESH = 0.14  # trigger STOP if bottom-center occupancy > this
fail_dir = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\failure_frames"
os.makedirs(fail_dir, exist_ok=True)

def path_hits_obstacle(grid, path, k=40):
    """Return True if the first k points of path intersect occupied cells."""
    for r, c in path[:k]:
        if grid[r, c] == 1:
            return True
    return False

# auto-range (respects TEST_ONLY)
exp_range = range(EXP_START, EXP_END) if not TEST_ONLY else range(EXP_START, min(EXP_START+6, EXP_END))

for exp in exp_range:
    exp_dir  = rf"{BASE_CORRIDOR}\exp{exp}"
    out_path = rf"{VID_DIR}\exp{exp}_avoidance.mp4"

    if not os.path.isdir(exp_dir):
        print("skip (missing):", exp_dir)
        continue
    if os.path.exists(out_path):
        print("✅ already done:", out_path)
        continue

    frames = sorted(glob.glob(rf"{exp_dir}\exp{exp}_img*.jpg"), key=numeric_key)
    if not frames:
        print("no frames in", exp_dir)
        continue

    first = cv2.imread(frames[0]); H, W = first.shape[:2]
    vw = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), 20.0, (W, H))
    print(f"exp{exp}: {len(frames)} frames")

    last_path = []
    for i, fp in enumerate(frames, 1):
        frame = cv2.imread(fp)
        if frame is None:
            continue

        # 1) YOLO → mask
        boxes = yolo_obstacles(frame, conf=DET_CONF)
        mask  = boxes_to_mask(frame.shape, boxes, inflate=INFLATE_PIXELS)

        # 2) (optional) adaptive center-depth probe
        if USE_DEPTH_PROBE and (i % PROBE_EVERY == 0):
            probe = depth_probe_mask(frame)
            mask  = cv2.bitwise_or(mask, probe)

        # 3) Occupancy grid (+ floor prior to suppress ceiling)
        grid = build_occupancy(mask, dilate_iter=2)
        floor_prior = np.zeros_like(grid, np.uint8); floor_prior[int(H*0.20):, :] = 1
        grid = cv2.bitwise_and(grid, floor_prior)

        # 4) Plan
        start = (int(H * 0.85), int(W * 0.5))
        goal  = (int(H * 0.15), int(W * 0.5))
        if (not last_path) or (i % REPLAN_EVERY == 0):
            last_path = astar(grid, start, goal)

        # 5) STOP rule if bottom-center is blocked
        bottom = grid[int(H*0.80):, int(W*0.40):int(W*0.60)]
        blocked = (bottom.mean() > STOP_BLOCKED_THRESH)
        if blocked:
            last_path = []  # don't draw a path if we must stop

        # 6) Draw overlays (tint occupancy so it's clear why we stopped/steered)
        vis = frame.copy()
        overlay = cv2.applyColorMap(mask, cv2.COLORMAP_JET)
        vis = cv2.addWeighted(vis, 0.75, overlay, 0.25, 0)

        for (x1, y1, x2, y2) in boxes:
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)
        for (r, c) in last_path:
            cv2.circle(vis, (c, r), 2, (0, 255, 0), -1)

        if blocked:
            cv2.putText(vis, "STOP: BLOCKED AHEAD", (10, H-20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        cv2.putText(vis, f"exp{exp} {i}/{len(frames)}  boxes:{len(boxes)} path:{len(last_path)}",
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        # 7) Save failure frames for quick debugging
        if last_path and path_hits_obstacle(grid, last_path):
            dbg = vis.copy()
            cv2.putText(dbg, "PATH HIT OBSTACLE", (10, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            cv2.imwrite(os.path.join(fail_dir, f"exp{exp}_frame{i:04d}_pathhit.png"), dbg)
        if blocked:
            cv2.imwrite(os.path.join(fail_dir, f"exp{exp}_frame{i:04d}_blocked.png"), vis)

        vw.write(vis)

    vw.release()
    print("✅ saved →", out_path)


------------------------------------------------------------------------

In [2]:
!pip install ultralytics --upgrade --quiet


In [3]:
# === CELL 1: IMPORTS ===
import os, re, glob, cv2, heapq, torch, numpy as np
from ultralytics import YOLO
from transformers import DPTForDepthEstimation, DPTImageProcessor


In [4]:
# === CELL 2: PATHS ===
BASE_CORRIDOR = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\3_Corridor\3_Corridor"
VID_DIR       = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos"
FAIL_DIR      = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\failure_frames"

os.makedirs(VID_DIR, exist_ok=True)
os.makedirs(FAIL_DIR, exist_ok=True)


In [5]:
# === CELL 3: MODEL SETUP ===
device = "cuda" if torch.cuda.is_available() else "cpu"
yolo = YOLO("yolov8n.pt")

_depth_ready = False
def _ensure_depth_model():
    global _depth_ready, depth_model, depth_proc
    if _depth_ready:
        return
    depth_model = DPTForDepthEstimation.from_pretrained("Intel/dpt-hybrid-midas").to(device)
    depth_proc  = DPTImageProcessor.from_pretrained("Intel/dpt-hybrid-midas")
    _depth_ready = True


In [6]:
# === CELL 4: PARAMETERS ===
EXP_START, EXP_END = 400, 733
TEST_ONLY          = False   # Set True for testing on fewer experiments

# Normal tuning
DET_CONF        = 0.32
MIN_AREA_FRAC   = 0.0015
MAX_AREA_FRAC   = 0.55
INFLATE_PIXELS  = 14
REPLAN_EVERY    = 4

# Depth probe
USE_DEPTH_PROBE  = True
PROBE_EVERY      = 1
PROBE_WIDTH_FRAC = 0.40
PROBE_TARGET_W   = 448
ADAPT_PERCENTILE = 65
DILATE_ITER      = 3

# Aggressive (early frame) tuning
AGG_FRAMES          = 60
AGG_DET_CONF        = 0.28
AGG_MIN_AREA_FRAC   = 0.0010
AGG_MAX_AREA_FRAC   = 0.60
AGG_INFLATE_PIXELS  = 12
AGG_PROBE_EVERY     = 1
AGG_PROBE_WIDTH     = 0.38
AGG_ADAPT_PERCENT   = 70

# Stop condition threshold
STOP_BLOCKED_THRESH = 0.14

# --- SEAM (vertical edge) fallback ---
SEAM_WIDTH_FRAC   = 0.40   # central band to analyze
SEAM_MIN_PEAK     = 0.12   # min normalized column-gradient peak
SEAM_MIN_GAP_PX   = 30     # min distance between the two peaks
SEAM_BRIGHT_MIN   = 110    # require bright surface inside band
SEAM_EDGE_DENSITY = 0.035  # require "low edge density" inside band
SEAM_DILATE_ITER  = 3

# --- LOOMING (optical flow) fallback ---
USE_FLOW          = True
FLOW_EVERY        = 2      # compute flow every N frames
FLOW_CENTER_W     = 0.32   # center band width for looming check
FLOW_MAG_THRESH   = 1.2    # min avg magnitude in center band
FLOW_RATIO_THRESH = 1.6    # center magnitude / side magnitude
FLOW_DILATE_ITER  = 3

# --- Box memory so detections don't blink ---
BOX_MEMORY_FRAMES = 8      # keep boxes this many frames with decay


In [7]:
# === CELL 5: HELPER FUNCTIONS ===
import collections

# --- Box memory (simple persistence) ---
class BoxMemory:
    def __init__(self, ttl=8):
        self.ttl = ttl
        self.items = collections.deque()  # (box, age)

    def update(self, new_boxes):
        # age existing
        aged = []
        for (b, age) in self.items:
            if age < self.ttl:
                aged.append((b, age+1))
        # add new as age=0
        for b in new_boxes:
            aged.append((b, 0))
        self.items = collections.deque(aged)

    def get(self):
        # return boxes with age-based inflate to keep visible
        return [b for (b, age) in self.items]

# --- Seam (vertical-plane) fallback ---
def seam_plane_mask(frame_bgr,
                    width_frac=SEAM_WIDTH_FRAC,
                    min_peak=SEAM_MIN_PEAK,
                    min_gap_px=SEAM_MIN_GAP_PX,
                    bright_min=SEAM_BRIGHT_MIN,
                    edge_density_max=SEAM_EDGE_DENSITY,
                    dilate_iter=SEAM_DILATE_ITER):
    H, W = frame_bgr.shape[:2]
    cx1 = int(W*(0.5 - width_frac/2))
    cx2 = int(W*(0.5 + width_frac/2))
    band = frame_bgr[:, cx1:cx2]
    gray = cv2.cvtColor(band, cv2.COLOR_BGR2GRAY)

    # column-wise gradient energy (Sobel X)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    col_energy = np.abs(gx).mean(axis=0)          # shape: [cx2-cx1]
    if col_energy.max() < 1e-6:
        return np.zeros((H, W), np.uint8)

    # normalize & find two peaks around the center
    ce = col_energy / (col_energy.max() + 1e-9)
    mid = len(ce)//2
    # strongest peak left side and right side
    left_idx  = np.argmax(ce[:mid]) if mid>4 else 0
    right_idx = np.argmax(ce[mid:]) + mid if mid>0 else len(ce)-1
    left_val, right_val = ce[left_idx], ce[right_idx]

    # quick texture/brightness check inside band
    edges = cv2.Canny(gray, 40, 120)
    edge_density = (edges > 0).mean()
    mean_bright  = gray.mean()

    ok_peaks = (left_val >= min_peak and right_val >= min_peak and (right_idx-left_idx) >= min_gap_px)
    ok_appearance = (edge_density <= edge_density_max and mean_bright >= bright_min)

    if ok_peaks and ok_appearance:
        m = np.zeros((H, W), np.uint8)
        m[:, cx1+left_idx:cx1+right_idx] = 255
        m = cv2.dilate(m, np.ones((3,3), np.uint8), iterations=dilate_iter)
        return m
    return np.zeros((H, W), np.uint8)

# --- Looming (optical-flow) STOP mask ---
_prev_gray = None
def looming_stop_mask(frame_bgr,
                      every=FLOW_EVERY,
                      center_w=FLOW_CENTER_W,
                      mag_thr=FLOW_MAG_THRESH,
                      ratio_thr=FLOW_RATIO_THRESH,
                      dilate_iter=FLOW_DILATE_ITER,
                      frame_idx=1):
    global _prev_gray
    if not USE_FLOW:
        return np.zeros(frame_bgr.shape[:2], np.uint8)

    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    if _prev_gray is None or (frame_idx % every) != 0:
        _prev_gray = gray
        return np.zeros(frame_bgr.shape[:2], np.uint8)

    flow = cv2.calcOpticalFlowFarneback(_prev_gray, gray, None,
                                        pyr_scale=0.5, levels=2, winsize=21,
                                        iterations=3, poly_n=5, poly_sigma=1.2, flags=0)
    _prev_gray = gray

    mag = np.sqrt(flow[...,0]**2 + flow[...,1]**2)
    H, W = gray.shape
    c1 = int(W*(0.5 - center_w/2)); c2 = int(W*(0.5 + center_w/2))
    center_mag = mag[:, c1:c2].mean()
    side_mag   = np.concatenate([mag[:, :c1], mag[:, c2:]], axis=1).mean() if c1>0 and c2<W else 1e-6
    ratio = center_mag / (side_mag + 1e-9)

    if center_mag >= mag_thr and ratio >= ratio_thr:
        m = np.zeros_like(gray, np.uint8)
        m[:, c1:c2] = 255
        m = cv2.dilate(m, np.ones((3,3), np.uint8), iterations=dilate_iter)
        return m
    return np.zeros_like(gray, np.uint8)

def yolo_obstacles(frame_bgr, conf=DET_CONF):
    """Detect obstacles using YOLO"""
    res = yolo.predict(source=frame_bgr, conf=conf, verbose=False,
                       device=0 if device=="cuda" else None)[0]
    H, W = frame_bgr.shape[:2]; A = H * W
    boxes = []
    if res.boxes is not None:
        xyxy = res.boxes.xyxy.cpu().numpy().astype(int)
        for (x1, y1, x2, y2) in xyxy:
            area = max(1, (x2-x1)*(y2-y1)) / A
            if area < MIN_AREA_FRAC or area > MAX_AREA_FRAC: continue
            boxes.append((x1, y1, x2, y2))
    return boxes


def boxes_to_mask(shape, boxes, inflate=INFLATE_PIXELS):
    """Convert YOLO boxes to binary mask"""
    H, W = shape[:2]; m = np.zeros((H, W), np.uint8)
    for (x1, y1, x2, y2) in boxes:
        x1i=max(0,x1-inflate); y1i=max(0,y1-inflate)
        x2i=min(W-1,x2+inflate); y2i=min(H-1,y2+inflate)
        m[y1i:y2i+1, x1i:x2i+1] = 255
    return m


def build_occupancy(mask, dilate_iter=2):
    """Generate occupancy grid"""
    grid = cv2.dilate(mask, np.ones((3,3), np.uint8), iterations=dilate_iter)
    return (grid>0).astype(np.uint8)


def astar(grid, start, goal):
    """A* Pathfinding"""
    H,W = grid.shape
    def h(a,b): return abs(a[0]-b[0])+abs(a[1]-b[1])
    openq=[(h(start,goal),0,start,None)]; came,cost,seen={}, {start:0}, set()
    nbrs=[(1,0),(-1,0),(0,1),(0,-1),(1,1),(1,-1),(-1,1),(-1,-1)]
    while openq:
        _,g,cur,parent = heapq.heappop(openq)
        if cur in seen: continue
        seen.add(cur); came[cur]=parent
        if cur==goal: break
        for dx,dy in nbrs:
            nx,ny = cur[0]+dx, cur[1]+dy
            if not (0<=nx<H and 0<=ny<W): continue
            if grid[nx,ny]==1: continue
            ng = g + (1.4 if dx and dy else 1.0)
            if ng < cost.get((nx,ny), 1e9):
                cost[(nx,ny)] = ng
                heapq.heappush(openq, (ng + h((nx,ny),goal), ng, (nx,ny), cur))
    if goal not in came: return []
    path,node=[],goal
    while node: path.append(node); node=came[node]
    return path[::-1]


In [8]:
# === CELL 6: DEPTH + EDGE FALLBACK ===

def depth_probe_mask(frame_bgr):
    """Adaptive center-ROI depth threshold"""
    _ensure_depth_model()
    H, W = frame_bgr.shape[:2]
    cx1 = int(W*(0.5 - PROBE_WIDTH_FRAC/2)); cx2 = int(W*(0.5 + PROBE_WIDTH_FRAC/2))
    roi = frame_bgr[:, cx1:cx2]
    h, w = roi.shape[:2]
    tw = PROBE_TARGET_W; th = int(h * (tw / max(1, w)))
    roi_small = cv2.resize(roi, (tw, th))
    rgb = cv2.cvtColor(roi_small, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        inputs = depth_proc(images=rgb, return_tensors="pt").to(device)
        d = depth_model(**inputs).predicted_depth.squeeze().detach().cpu().numpy()
    d = (d - d.min()) / (d.max() - d.min() + 1e-8)
    d8 = 255 - (d * 255).astype(np.uint8)
    d8 = cv2.resize(d8, (cx2-cx1, H), interpolation=cv2.INTER_LINEAR)
    thr = np.percentile(d8, ADAPT_PERCENTILE)
    mask = (d8 >= thr).astype(np.uint8) * 255
    full = np.zeros((H, W), np.uint8); full[:, cx1:cx2] = mask
    full = cv2.dilate(full, np.ones((3,3), np.uint8), iterations=DILATE_ITER)
    return full


def edge_plane_mask(frame_bgr,
                    width_frac=PROBE_WIDTH_FRAC,
                    edge_min_density=0.006,   # ↓ allow fewer edges
                    var_max=160.0,            # ↓ allow very low texture
                    bright_min=70,            # ↓ allow dim scenes
                    p70_min=95,               # percentile check for “bright-ish”
                    canny_lo=20, canny_hi=80,
                    close_k=5, dilate_iter=3):
    """
    Marks the center strip as obstacle if it looks like a flat wall/cabinet:
      - very low edge density
      - low gray-level variance (textureless)
      - reasonably bright (mean or 70th percentile)
    Returns a 0/255 mask the same size as the frame.
    """
    H, W = frame_bgr.shape[:2]
    cx1 = int(W * (0.5 - width_frac / 2.0))
    cx2 = int(W * (0.5 + width_frac / 2.0))
    roi = frame_bgr[:, cx1:cx2]

    # normalize a bit to fight vignetting
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)

    # edge + texture
    edges = cv2.Canny(gray, canny_lo, canny_hi)
    edge_density = (edges > 0).mean()
    var = float(gray.var())
    mean_bright = float(gray.mean())
    p70 = float(np.percentile(gray, 70))

    # decision: very low edges & low texture & bright enough
    is_bright = (mean_bright >= bright_min) or (p70 >= p70_min)
    looks_flat = (edge_density < edge_min_density) and (var <= var_max)

    if looks_flat and is_bright:
        m = np.zeros((H, W), np.uint8)
        m[:, cx1:cx2] = 255
        # clean the band so the planner respects it robustly
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((close_k, close_k), np.uint8), iterations=1)
        m = cv2.dilate(m, np.ones((3,3), np.uint8), iterations=dilate_iter)
        return m

    return np.zeros((H, W), np.uint8)

 

TEST_ONLY = True
EXP_START = 400
EXP_END = 404


In [14]:
# === CELL 7: MAIN LOOP (FINAL + HYSTERESIS + MEMORY + COOLDOWN) ===

import re, os, glob, cv2, numpy as np

# --- helper for path hits ---
def path_hits_obstacle(grid, path, k=40):
    for r, c in path[:k]:
        if grid[r, c] == 1:
            return True
    return False

# === Ensure dirs & safe defaults ===
FAIL_DIR = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\failure_frames"
os.makedirs(FAIL_DIR, exist_ok=True)

STOP_BLOCKED_THRESH = 0.14
HOLD_AFTER_OBS_FRAMES = 12   # cooldown after obstacle (~0.6s at 20 FPS)

# === Optional tracker / memory ===
try:
    from boxmot.tracker_zoo import create_tracker
    tracker = create_tracker("bytetrack", reid=False, device="cuda" if device == "cuda" else "cpu")
    USE_TRACKER = True
    print("✅ Using ByteTrack for steady detections")
except Exception:
    print("⚠️ Tracker unavailable, using BoxMemory fallback.")
    USE_TRACKER = False

try:
    box_mem
except NameError:
    try:
        box_mem = BoxMemory(ttl=12, iou_match=0.5)
    except Exception:
        box_mem = None

# === CONF + AREA hysteresis ===
CONF_KEEP, CONF_DROP = 0.4, 0.25
AREA_KEEP_FRAC, AREA_DROP_FRAC = 0.002, 0.0012

def apply_conf_area_hysteresis(dets, frame_shape):
    """Apply both conf + area hysteresis."""
    H, W = frame_shape[:2]; A = H * W
    out = []
    for d in dets:
        x1, y1, x2, y2, cls, conf = d
        area_frac = ((x2 - x1) * (y2 - y1)) / A
        if conf >= CONF_KEEP or area_frac >= AREA_KEEP_FRAC:
            out.append(d)
        elif conf < CONF_DROP or area_frac < AREA_DROP_FRAC:
            continue
        else:
            out.append(d)
    return out

# === Strong morphology for masks ===
MORPH_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
MORPH_CLOSE_ITERS, MORPH_DILATE_ITERS = 2, 2

def boxes_to_mask_strong(shape, boxes_xyxy, inflate=12):
    H, W = shape[:2]
    m = np.zeros((H, W), np.uint8)
    for (x1,y1,x2,y2,*_) in boxes_xyxy:
        x1i=max(0,x1-inflate); y1i=max(0,y1-inflate)
        x2i=min(W-1,x2+inflate); y2i=min(H-1,y2+inflate)
        m[y1i:y2i+1, x1i:x2i+1] = 255
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, MORPH_KERNEL, iterations=MORPH_CLOSE_ITERS)
    m = cv2.dilate(m, MORPH_KERNEL, iterations=MORPH_DILATE_ITERS)
    return m

# === Main experiment loop ===
exp_range = range(EXP_START, EXP_END) if not TEST_ONLY else range(EXP_START, min(EXP_START + 6, EXP_END))

for exp in exp_range:
    exp_dir = rf"{BASE_CORRIDOR}\exp{exp}"
    out_path = rf"{VID_DIR}\exp{exp}_avoidance.mp4"

    if os.path.exists(out_path) and os.path.getsize(out_path) > 100 * 1024:
        print(f"✅ Skipping (already processed): {out_path}")
        continue
    if not os.path.isdir(exp_dir):
        print(f"⏭️ Missing dir: {exp_dir}")
        continue

    frames = sorted(
        glob.glob(rf"{exp_dir}\*.jpg") + glob.glob(rf"{exp_dir}\*.jpeg") + glob.glob(rf"{exp_dir}\*.png"),
        key=lambda x: int(re.search(r"(\d+)(?=\.[^.]+$)", x).group(1)) if re.search(r"(\d+)(?=\.[^.]+$)", x) else 0
    )

    if not frames:
        print(f"⚠️ No frames in {exp_dir}")
        continue

    print(f"▶️ exp{exp}: {len(frames)} frames")

    first = cv2.imread(frames[0]); H, W = first.shape[:2]
    vw = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), 20.0, (W, H))

    last_path = []
    hold_counter = 0

    for i, fp in enumerate(frames, 1):
        frame = cv2.imread(fp)
        if frame is None:
            continue

        # --- YOLO detection full ---
        dets_full = yolo_obstacles_full(frame, conf=DET_CONF)
        dets_full = apply_conf_area_hysteresis(dets_full, frame.shape)

        # --- Tracking / memory ---
        if USE_TRACKER:
            from numpy import array
            det_arr = array([[x1,y1,x2,y2,conf,cls] for (x1,y1,x2,y2,cls,conf) in dets_full], dtype=float)
            online_targets = tracker.update(det_arr, frame)
            boxes_for_mask = [(int(x1),int(y1),int(x2),int(y2),int(cls),float(conf)) for x1,y1,x2,y2,_,cls,conf in online_targets]
        else:
            if box_mem is not None:
                boxes_for_mask = box_mem.update(dets_full)
            else:
                boxes_for_mask = dets_full

        # --- Build YOLO mask ---
        mask = boxes_to_mask_strong(frame.shape, boxes_for_mask, inflate=INFLATE_PIXELS)

        # --- Add depth probe ---
        if USE_DEPTH_PROBE and (i % PROBE_EVERY == 0):
            mask = cv2.bitwise_or(mask, depth_probe_mask(frame))

        # --- Add seam/edge fallback ---
        try:
            seam_mask = seam_plane_mask(frame)
        except NameError:
            seam_mask = edge_plane_mask(frame)
        mask = cv2.bitwise_or(mask, seam_mask)

        # --- Optical flow stop (if exists) ---
        try:
            flow_mask = looming_stop_mask(frame, frame_idx=i)
            mask = cv2.bitwise_or(mask, flow_mask)
        except NameError:
            flow_mask = np.zeros(frame.shape[:2], np.uint8)

        # --- COOLDOWN logic ---
        mask_nonzero = (np.count_nonzero(mask) > 0)
        seen_any = (len(boxes_for_mask) > 0) or seam_mask.any() or flow_mask.any() or mask_nonzero
        hold_counter = HOLD_AFTER_OBS_FRAMES if seen_any else max(0, hold_counter - 1)

        # --- Occupancy + A* ---
        grid = build_occupancy(mask, dilate_iter=2)
        start = (int(H * 0.85), int(W * 0.5))
        goal  = (int(H * 0.15), int(W * 0.5))
        if not last_path or (i % REPLAN_EVERY == 0):
            last_path = astar(grid, start, goal)

        # --- STOP checks ---
        bottom = grid[int(H*0.80):, int(W*0.40):int(W*0.60)]
        blocked = (bottom.mean() > STOP_BLOCKED_THRESH)
        if not blocked and len(boxes_for_mask) == 0 and (mask.mean() < 0.01 * 255):
            blocked = True
        if hold_counter > 0:
            blocked = True
        if blocked:
            last_path = []

        # --- Visualization ---
        vis = frame.copy()

        if seam_mask.any():
            red_overlay = np.zeros_like(vis); red_overlay[:,:,2] = seam_mask
            vis = cv2.addWeighted(vis, 1.0, red_overlay, 0.3, 0)
        if flow_mask.any():
            blu_overlay = np.zeros_like(vis); blu_overlay[:,:,0] = flow_mask
            vis = cv2.addWeighted(vis, 1.0, blu_overlay, 0.25, 0)

        for (x1,y1,x2,y2,*_) in boxes_for_mask:
            cv2.rectangle(vis,(x1,y1),(x2,y2),(0,0,255),2)

        for (r,c) in last_path:
            cv2.circle(vis,(c,r),2,(0,255,0),-1)

        if blocked:
            msg = "STOP: BLOCKED AHEAD" if hold_counter == 0 else "STOP: COOLDOWN HOLD"
            cv2.putText(vis,msg,(10,H-20),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,0,255),2)

        if last_path and path_hits_obstacle(grid,last_path):
            fname = os.path.join(FAIL_DIR,f"exp{exp}_frame{i:04d}_pathhit.png")
            overlay = cv2.applyColorMap(mask, cv2.COLORMAP_JET)
            debug = cv2.addWeighted(vis,0.6,overlay,0.4,0)
            cv2.putText(debug,"PATH HIT OBSTACLE",(10,45),
                        cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,0,255),2)
            cv2.imwrite(fname,debug)

        cv2.putText(vis,f"exp{exp} frame {i}/{len(frames)} boxes:{len(boxes_for_mask)} path:{len(last_path)}",
                    (10,25),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)
        vw.write(vis)

    vw.release()
    print(f"✅ Saved → {out_path}")


⚠️ Tracker unavailable, using BoxMemory fallback.
✅ Skipping (already processed): C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp400_avoidance.mp4
✅ Skipping (already processed): C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp401_avoidance.mp4
✅ Skipping (already processed): C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp402_avoidance.mp4
✅ Skipping (already processed): C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp403_avoidance.mp4


In [ ]:
# === AUTO SMOKE TEST FOR WHITE WALL/CABINET SCENES ===
import os, re, glob, cv2, numpy as np

CANDIDATE_EXPS    = [717, 700, 728, 731]     # change or extend this list
MAX_LOOK_FRAMES   = 80                       # scan up to N frames per exp for a white wall
WINDOW_BEFORE     = 20                       # how many frames before the hit to include
MAX_TEST_FRAMES   = 60                       # total frames in the preview video
TEST_FPS          = 20.0
EDGE_MIN_DENS     = 0.02                     # “flat” if below this
BRIGHT_MIN        = 110                      # mean gray in center ROI must exceed this

def list_frames(exp):
    exp_dir = rf"{BASE_CORRIDOR}\exp{exp}"
    frames = sorted(
        glob.glob(rf"{exp_dir}\*.jpg") +
        glob.glob(rf"{exp_dir}\*.jpeg") +
        glob.glob(rf"{exp_dir}\*.png"),
        key=lambda x: int(re.search(r"(\d+)(?=\.[^.]+$)", x).group(1)) if re.search(r"(\d+)(?=\.[^.]+$)", x) else 0
    )
    return exp_dir, frames

def is_bright_flat_center(img, width_frac=0.34):
    """Return True if center band is bright and has very few edges (white wall/cabinet)."""
    H, W = img.shape[:2]
    cx1 = int(W * (0.5 - width_frac/2))
    cx2 = int(W * (0.5 + width_frac/2))
    roi = img[:, cx1:cx2]
    g   = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(g, 40, 120)
    edge_density = (edges > 0).mean()
    mean_bright  = g.mean()
    return (edge_density < EDGE_MIN_DENS) and (mean_bright >= BRIGHT_MIN)

# --- find a suitable experiment & starting index ---
chosen_exp, start_idx = None, None
chosen_frames = []
for exp in CANDIDATE_EXPS:
    exp_dir, frames = list_frames(exp)
    if not frames:
        print(f"⚠️ No frames in {exp_dir}")
        continue
    # probe the first MAX_LOOK_FRAMES (or all, if fewer)
    probe = frames[:min(MAX_LOOK_FRAMES, len(frames))]
    hit = None
    for i, fp in enumerate(probe):
        im = cv2.imread(fp)
        if im is None: 
            continue
        if is_bright_flat_center(im):
            hit = i
            break
    if hit is not None:
        chosen_exp   = exp
        # build a window around the hit index
        s = max(0, hit - WINDOW_BEFORE)
        e = min(len(frames), s + MAX_TEST_FRAMES)
        chosen_frames = frames[s:e]
        start_idx = s
        print(f"✅ Found bright/flat center in exp{exp} at frame {hit}; taking {len(chosen_frames)} frames [{s}:{e}]")
        break

assert chosen_frames, "Could not find a candidate white-wall/cabinet scene in the candidate experiments."

# --- write preview ---
first = cv2.imread(chosen_frames[0]); H, W = first.shape[:2]
out_path = os.path.join(VID_DIR, f"smoketest_whitewall_exp{chosen_exp}.mp4")
vw = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), TEST_FPS, (W, H))
print(f"🎬 Writing preview → {out_path}")

last_path = []
for i, fp in enumerate(chosen_frames, 1):
    frame = cv2.imread(fp)
    if frame is None: 
        continue

    # 1) YOLO → mask
    boxes = yolo_obstacles(frame, conf=DET_CONF)
    mask  = boxes_to_mask(frame.shape, boxes, inflate=INFLATE_PIXELS)

    # 2) depth probe sometimes
    if USE_DEPTH_PROBE and (i % PROBE_EVERY == 0):
        mask = cv2.bitwise_or(mask, depth_probe_mask(frame))

    # 3) seam/edge fallback for white walls/cabinets
    try:
        seam_mask = seam_plane_mask(frame)   # if you added the seam-based version
    except NameError:
        seam_mask = edge_plane_mask(frame)   # fallback to edge-plane version
    mask = cv2.bitwise_or(mask, seam_mask)

    # 4) occupancy + A*
    grid  = build_occupancy(mask, dilate_iter=2)
    start = (int(H*0.85), int(W*0.5))
    goal  = (int(H*0.15), int(W*0.5))
    if not last_path or (i % REPLAN_EVERY == 0):
        last_path = astar(grid, start, goal)

    # 5) STOP if bottom band blocked
    bottom  = grid[int(H*0.80):, int(W*0.40):int(W*0.60)]
    blocked = (bottom.mean() > STOP_BLOCKED_THRESH)
    if blocked:
        last_path = []

    # 6) visualize
    vis = frame.copy()
    if seam_mask.any():
        red = np.zeros_like(vis); red[:,:,2] = seam_mask
        vis = cv2.addWeighted(vis, 1.0, red, 0.35, 0)
    for (x1,y1,x2,y2) in boxes:
        cv2.rectangle(vis,(x1,y1),(x2,y2),(0,0,255),2)
    for (r,c) in last_path:
        cv2.circle(vis,(c,r),2,(0,255,0),-1)

    cv2.putText(vis, f"exp{chosen_exp} test {start_idx+i}/{start_idx+len(chosen_frames)} "
                     f"boxes:{len(boxes)} path:{len(last_path)}",
                (10,25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    if blocked:
        cv2.putText(vis, "STOP: BLOCKED AHEAD", (10, H-20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

    vw.write(vis)

vw.release()
print("✅ Preview saved:", out_path)

# --- one-frame debug montage at the midpoint ---
mid_fp = chosen_frames[len(chosen_frames)//2]
frm = cv2.imread(mid_fp)
boxes_mid = yolo_obstacles(frm, conf=DET_CONF)
m1 = boxes_to_mask(frm.shape, boxes_mid, inflate=INFLATE_PIXELS)
m2 = depth_probe_mask(frm) if USE_DEPTH_PROBE else np.zeros(frm.shape[:2], np.uint8)
try:
    m3 = seam_plane_mask(frm)
except NameError:
    m3 = edge_plane_mask(frm)

ov1 = cv2.addWeighted(frm, 1.0, cv2.cvtColor(m1, cv2.COLOR_GRAY2BGR), 0.35, 0)
ov2 = cv2.addWeighted(frm, 1.0, cv2.cvtColor(m2, cv2.COLOR_GRAY2BGR), 0.35, 0)
ov3 = cv2.addWeighted(frm, 1.0, cv2.cvtColor(m3, cv2.COLOR_GRAY2BGR), 0.35, 0)
for img, txt in [(ov1,"YOLO mask"), (ov2,"Depth probe"), (ov3,"Wall/cabinet fallback")]:
    cv2.putText(img, txt, (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8,(255,255,255),2)
montage = np.hstack([ov1, ov2, ov3])
dbg_path = os.path.join(VID_DIR, f"smoketest_whitewall_exp{chosen_exp}_montage.jpg")
cv2.imwrite(dbg_path, montage)
print("🖼️ Debug montage saved:", dbg_path)


In [ ]:
# === CELL 8: MERGE DEMO ===
import os, cv2, numpy as np, glob

demo_out = os.path.join(VID_DIR, "hospital_demo_final12.mp4")
picks = sorted(glob.glob(os.path.join(VID_DIR, "exp*_avoidance.mp4")))

if not picks:
    print("⚠️ No processed videos found.")
else:
    cap0 = cv2.VideoCapture(picks[0])
    fps = cap0.get(cv2.CAP_PROP_FPS) or 20.0
    W = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap0.release()

    writer = cv2.VideoWriter(demo_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    SECONDS_PER_CLIP = 6

    for p in picks:
        exp_id = os.path.basename(p).split("_")[0]
        print(f"🎞️ Adding {exp_id} to demo")
        
        title = np.zeros((H, W, 3), np.uint8)
        cv2.putText(title, f"{exp_id}: YOLO + Depth + Edge Path Planning",
                    (30, int(H*0.55)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        for _ in range(int(fps*0.8)): writer.write(title)

        cap = cv2.VideoCapture(p)
        max_frames = int(fps * SECONDS_PER_CLIP)
        count = 0
        while count < max_frames:
            ret, fr = cap.read()
            if not ret: break
            writer.write(fr)
            count += 1
        cap.release()

    writer.release()
    print(f"✅ Demo merged → {demo_out}")


🎞️ Adding exp400 to demo
🎞️ Adding exp401 to demo
🎞️ Adding exp402 to demo
🎞️ Adding exp403 to demo
🎞️ Adding exp405 to demo
🎞️ Adding exp406 to demo
🎞️ Adding exp407 to demo
🎞️ Adding exp408 to demo
🎞️ Adding exp409 to demo
🎞️ Adding exp410 to demo
🎞️ Adding exp411 to demo
🎞️ Adding exp412 to demo
🎞️ Adding exp413 to demo
🎞️ Adding exp414 to demo
🎞️ Adding exp415 to demo
🎞️ Adding exp416 to demo
🎞️ Adding exp417 to demo
🎞️ Adding exp418 to demo
🎞️ Adding exp419 to demo
🎞️ Adding exp420 to demo
🎞️ Adding exp421 to demo
🎞️ Adding exp422 to demo
🎞️ Adding exp423 to demo
🎞️ Adding exp424 to demo
🎞️ Adding exp425 to demo
🎞️ Adding exp426 to demo
🎞️ Adding exp427 to demo
🎞️ Adding exp428 to demo
🎞️ Adding exp430 to demo
🎞️ Adding exp431 to demo
🎞️ Adding exp432 to demo
🎞️ Adding exp433 to demo
🎞️ Adding exp434 to demo
🎞️ Adding exp436 to demo
🎞️ Adding exp437 to demo
🎞️ Adding exp439 to demo
🎞️ Adding exp440 to demo
🎞️ Adding exp441 to demo
🎞️ Adding exp442 to demo
🎞️ Adding exp443 to demo


In [ ]:
# === TEST CELL: run on 5 clips + summary + mini demo ===
import os, re, glob, cv2, numpy as np
from collections import defaultdict

# ---- choose 5 experiments to test ----
TEST_EXPS = [400, 401, 700, 717, 728]  # edit if some are missing
MAX_FRAMES_PER_CLIP = 220              # cap per exp to keep it quick (≈11s at 20fps)
OVERWRITE = True                       # re-generate test videos
TEST_SUFFIX = "_avoidance_TEST.mp4"    # output naming

# ---- safety defaults if not defined earlier ----
try: VID_DIR
except NameError:
    VID_DIR = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos"
    os.makedirs(VID_DIR, exist_ok=True)

try: BASE_CORRIDOR
except NameError:
    BASE_CORRIDOR = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\3_Corridor\3_Corridor"

try: STOP_BLOCKED_THRESH
except NameError:
    STOP_BLOCKED_THRESH = 0.14

try: HOLD_AFTER_OBS_FRAMES
except NameError:
    HOLD_AFTER_OBS_FRAMES = 12

# Small helper
def numeric_key(p):
    m = re.search(r"(\d+)(?=\.[^.]+$)", p)
    return int(m.group(1)) if m else 0

# Stats store
stats = defaultdict(dict)
test_outputs = []

for exp in TEST_EXPS:
    exp_dir  = rf"{BASE_CORRIDOR}\exp{exp}"
    out_path = rf"{VID_DIR}\exp{exp}{TEST_SUFFIX}"

    if not os.path.isdir(exp_dir):
        print(f"⏭️ Skip (missing directory): {exp_dir}")
        continue

    if (not OVERWRITE) and os.path.exists(out_path) and os.path.getsize(out_path) > 100*1024:
        print(f"✅ Skipping (already tested): {out_path}")
        test_outputs.append(out_path)
        continue

    frames = sorted(
        glob.glob(rf"{exp_dir}\*.jpg") +
        glob.glob(rf"{exp_dir}\*.jpeg") +
        glob.glob(rf"{exp_dir}\*.png"),
        key=numeric_key
    )
    if not frames:
        print(f"⚠️ No frames in {exp_dir}")
        continue

    # Init writer
    first = cv2.imread(frames[0]); H, W = first.shape[:2]
    vw = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), 20.0, (W, H))

    # Per-clip counters
    n_frames = 0
    n_box_frames = 0
    n_blocked_frames = 0
    n_cooldown_frames = 0
    sum_boxes = 0

    last_path = []
    hold_counter = 0

    print(f"▶️ Testing exp{exp}: {len(frames)} frames (cap {MAX_FRAMES_PER_CLIP})")
    for i, fp in enumerate(frames, 1):
        if n_frames >= MAX_FRAMES_PER_CLIP:
            break

        frame = cv2.imread(fp)
        if frame is None:
            continue

        # --- 1) YOLO detection ---
        boxes = yolo_obstacles(frame, conf=DET_CONF)
        sum_boxes += len(boxes)
        if len(boxes) > 0:
            n_box_frames += 1

        # --- YOLO mask ---
        mask = boxes_to_mask(frame.shape, boxes, inflate=INFLATE_PIXELS)

        # --- 2) Adaptive depth probe (center strip) ---
        if USE_DEPTH_PROBE and (i % PROBE_EVERY == 0):
            mask = cv2.bitwise_or(mask, depth_probe_mask(frame))

        # --- 3) White wall/cabinet fallback (seam/bright plane) ---
        try:
            seam_mask = seam_plane_mask(frame)
        except NameError:
            seam_mask = edge_plane_mask(frame)
        mask = cv2.bitwise_or(mask, seam_mask)

        # --- 4) Looming STOP mask (optical flow), if available ---
        try:
            flow_mask = looming_stop_mask(frame, frame_idx=i)
            mask = cv2.bitwise_or(mask, flow_mask)
        except NameError:
            flow_mask = np.zeros(frame.shape[:2], np.uint8)

        # Cooldown logic
        mask_nonzero = (np.count_nonzero(mask) > 0)
        seen_any_obstacle = (len(boxes) > 0) or seam_mask.any() or flow_mask.any() or mask_nonzero
        if seen_any_obstacle:
            hold_counter = HOLD_AFTER_OBS_FRAMES
        else:
            hold_counter = max(0, hold_counter - 1)

        # --- 5) Occupancy + path planning ---
        grid  = build_occupancy(mask, dilate_iter=2)
        start = (int(H * 0.85), int(W * 0.5))
        goal  = (int(H * 0.15), int(W * 0.5))
        if not last_path or (i % REPLAN_EVERY == 0):
            last_path = astar(grid, start, goal)

        # --- 6) STOP rules ---
        bottom = grid[int(H*0.80):, int(W*0.40):int(W*0.60)]
        blocked = (bottom.mean() > STOP_BLOCKED_THRESH)

        if not blocked and len(boxes) == 0 and (mask.mean() < 0.01 * 255):
            blocked = True  # emergency brake

        if hold_counter > 0:
            blocked = True
            n_cooldown_frames += 1

        if blocked:
            last_path = []
            n_blocked_frames += 1

        # --- 7) Visualization ---
        vis = frame.copy()
        if seam_mask.any():
            red_overlay = np.zeros_like(vis); red_overlay[:, :, 2] = seam_mask
            vis = cv2.addWeighted(vis, 1.0, red_overlay, 0.30, 0)
        if flow_mask.any():
            blu_overlay = np.zeros_like(vis); blu_overlay[:, :, 0] = flow_mask
            vis = cv2.addWeighted(vis, 1.0, blu_overlay, 0.25, 0)

        for (x1, y1, x2, y2) in boxes:
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)

        for (r, c) in last_path:
            cv2.circle(vis, (c, r), 2, (0, 255, 0), -1)

        if blocked:
            msg = "STOP: BLOCKED AHEAD" if hold_counter == 0 else "STOP: RECENT OBSTACLE (COOLDOWN)"
            cv2.putText(vis, msg, (10, H-20), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        cv2.putText(vis, f"exp{exp} f{i}/{len(frames)} boxes:{len(boxes)} path:{len(last_path)}",
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        vw.write(vis)
        n_frames += 1

    vw.release()
    test_outputs.append(out_path)

    # Save stats
    stats[exp] = {
        "frames": n_frames,
        "box_frames": n_box_frames,
        "blocked_frames": n_blocked_frames,
        "cooldown_frames": n_cooldown_frames,
        "avg_boxes_per_frame": (sum_boxes / max(1, n_frames))
    }

# ---- Print summary table ----
if stats:
    print("\n=== TEST SUMMARY (5 clips) ===")
    print(f"{'exp':>6}  {'frames':>7}  {'boxes%':>7}  {'blocked%':>9}  {'cooldown%':>9}  {'avg_boxes':>9}")
    for exp, s in stats.items():
        f = s["frames"]
        boxes_pct   = 100.0 * s["box_frames"]     / max(1, f)
        blocked_pct = 100.0 * s["blocked_frames"] / max(1, f)
        cool_pct    = 100.0 * s["cooldown_frames"]/ max(1, f)
        print(f"{exp:>6}  {f:7d}  {boxes_pct:7.1f}  {blocked_pct:9.1f}  {cool_pct:9.1f}  {s['avg_boxes_per_frame']:9.2f}")

# ---- Merge a tiny demo from these 5 test outputs ----
if test_outputs:
    demo_out = os.path.join(VID_DIR, "hospital_demo_test5.mp4")
    # probe size/fps
    cap0=None
    for p in test_outputs:
        if os.path.exists(p):
            cap0 = cv2.VideoCapture(p)
            if cap0.isOpened(): break
    if cap0 is not None and cap0.isOpened():
        fps = cap0.get(cv2.CAP_PROP_FPS) or 20.0
        W   = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
        H   = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap0.release()

        writer = cv2.VideoWriter(demo_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
        SECONDS_PER_CLIP = 5

        for p in test_outputs:
            exp_id = os.path.basename(p).split("_")[0]
            title = np.zeros((H, W, 3), np.uint8)
            cv2.putText(title, f"{exp_id}: Test Pipeline (YOLO + Depth + Seam + A*)",
                        (30, int(H*0.55)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
            for _ in range(int(fps*0.6)): writer.write(title)

            cap = cv2.VideoCapture(p)
            max_frames = int(fps * SECONDS_PER_CLIP)
            n = 0
            while n < max_frames:
                ret, fr = cap.read()
                if not ret: break
                writer.write(fr); n += 1
            cap.release()

        writer.release()
        print(f"\n🎬 Merged test demo → {demo_out}")
    else:
        print("⚠️ Could not open any test outputs to create a demo.")
else:
    print("⚠️ No test outputs generated.")


In [9]:
# === CELL: BOX MEMORY (IoU + class + TTL) ===
import math

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2-ix1), max(0, iy2-iy1)
    inter = iw*ih
    area_a = max(1,(ax2-ax1)*(ay2-ay1))
    area_b = max(1,(bx2-bx1)*(by2-by1))
    return inter / (area_a + area_b - inter + 1e-9)

class BoxMemory:
    """
    Keeps boxes for N frames after they vanish.
    Matches by IoU & class. Updates bbox and smoothed confidence on match.
    """
    def __init__(self, ttl=12, iou_match=0.5, conf_decay=0.92):
        self.ttl = ttl
        self.iou_match = iou_match
        self.conf_decay = conf_decay
        self.tracks = []  # list of dicts: {box, cls, conf, life}

    def update(self, detections):
        """
        detections: list of (x1,y1,x2,y2, cls_id, conf)
        returns: stabilized list of (x1,y1,x2,y2, cls_id, conf)
        """
        used = [False]*len(detections)

        # 1) match existing tracks
        for t in self.tracks:
            best_j, best_iou = -1, 0.0
            for j, det in enumerate(detections):
                if used[j]: continue
                bx,by,bx2,by2, c, cf = det
                if t["cls"] is not None and c is not None and t["cls"] != c:
                    continue
                i = iou_xyxy((bx,by,bx2,by2), t["box"])
                if i > best_iou:
                    best_iou, best_j = i, j
            if best_j >= 0 and best_iou >= self.iou_match:
                # update track
                bx,by,bx2,by2, c, cf = detections[best_j]
                t["box"]  = (bx,by,bx2,by2)
                t["cls"]  = c
                t["conf"] = max(cf, self.conf_decay * t["conf"] + (1-self.conf_decay) * cf)
                t["life"] = self.ttl
                used[best_j] = True
            else:
                # no match; decay life/conf
                t["life"] -= 1
                t["conf"] *= self.conf_decay

        # 2) add new unmatched detections as tracks
        for j, det in enumerate(detections):
            if used[j]: continue
            bx,by,bx2,by2, c, cf = det
            self.tracks.append({"box":(bx,by,bx2,by2), "cls":c, "conf":cf, "life":self.ttl})

        # 3) purge dead
        self.tracks = [t for t in self.tracks if t["life"] > 0 and t["conf"] > 0.05]

        # 4) output
        return [(t["box"][0], t["box"][1], t["box"][2], t["box"][3], t["cls"], t["conf"]) for t in self.tracks]


In [10]:
def yolo_obstacles_full(frame_bgr, conf=0.25):
    """Returns list of (x1,y1,x2,y2, cls_id, conf_float) with class & confidence."""
    res = yolo.predict(source=frame_bgr, conf=conf, verbose=False,
                       device=0 if device=="cuda" else None)[0]
    out = []
    if res.boxes is not None:
        xyxy = res.boxes.xyxy.cpu().numpy().astype(int)
        clss = res.boxes.cls.cpu().numpy().astype(int) if res.boxes.cls is not None else None
        confs= res.boxes.conf.cpu().numpy().astype(float) if res.boxes.conf is not None else None
        for k, (x1,y1,x2,y2) in enumerate(xyxy):
            cid  = int(clss[k])  if clss is not None else -1
            cval = float(confs[k]) if confs is not None else 1.0
            out.append((x1,y1,x2,y2, cid, cval))
    return out


In [11]:
# === CELL: CONFIDENCE HYSTERESIS ===
# show if score >= CONF_KEEP, hide only if score < CONF_DROP
CONF_KEEP = 0.40
CONF_DROP = 0.25

def apply_conf_hysteresis(dets_with_conf, prev_kept_ids=None):
    """
    dets_with_conf: list of (x1,y1,x2,y2, cls, conf)
    prev_kept_ids:  optional set of indices from previous frame to bias keeping
    returns filtered list
    """
    keep = []
    for d in dets_with_conf:
        x1,y1,x2,y2, c, cf = d
        if cf >= CONF_KEEP:
            keep.append(d)
        elif cf < CONF_DROP:
            # drop unless we match a previous-kept box via IoU in BoxMemory (handled by BoxMemory)
            continue
        else:
            # mid band: keep only if BoxMemory will keep it (so here we pass it through)
            keep.append(d)
    return keep


In [12]:
# === CELL: AREA HYSTERESIS ===
AREA_KEEP_FRAC = 0.002   # must be >= this to enter
AREA_DROP_FRAC = 0.0012  # once kept, only drop if it shrinks below this

def apply_area_hysteresis(dets, frame_shape, prev_boxes=None):
    """
    dets: list of (x1,y1,x2,y2, cls, conf)
    prev_boxes: stabilized list from last frame (to allow dropping slowly)
    """
    H,W = frame_shape[:2]; A = float(H*W)
    kept = []
    for d in dets:
        x1,y1,x2,y2,c,cf = d
        area_frac = max(1,(x2-x1)*(y2-y1))/A
        if area_frac >= AREA_KEEP_FRAC:
            kept.append(d)
        elif area_frac < AREA_DROP_FRAC:
            # drop unless memory keeps it (memory applied after this)
            continue
        else:
            # mid band passes through to memory stage
            kept.append(d)
    return kept


In [13]:
# === CELL: STRONGER MORPHOLOGY ===
# Use bigger kernel / more iterations to close mask holes
MORPH_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
MORPH_CLOSE_ITERS = 2
MORPH_DILATE_ITERS = 2

def boxes_to_mask_strong(shape, boxes_xyxy, inflate=12):
    H, W = shape[:2]
    m = np.zeros((H, W), np.uint8)
    for (x1,y1,x2,y2, *_rest) in boxes_xyxy:  # allow tuples w/ cls/conf
        x1i = max(0, x1-inflate); y1i = max(0, y1-inflate)
        x2i = min(W-1, x2+inflate); y2i = min(H-1, y2+inflate)
        m[y1i:y2i+1, x1i:x2i+1] = 255
    # close small gaps then dilate a bit
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, MORPH_KERNEL, iterations=MORPH_CLOSE_ITERS)
    m = cv2.dilate(m, MORPH_KERNEL, iterations=MORPH_DILATE_ITERS)
    return m


In [15]:
# === TEST CELL B: QUICK DEMO FROM EXISTING OUTPUTS ===
import os, cv2, numpy as np, glob

demo_out = os.path.join(VID_DIR, "smoketest_from_outputs.mp4")
picks = sorted(glob.glob(os.path.join(VID_DIR, "exp*_avoidance.mp4")))[:6]  # first 6

if not picks:
    print("⚠️ No processed videos found in", VID_DIR)
else:
    cap0 = cv2.VideoCapture(picks[0])
    fps = cap0.get(cv2.CAP_PROP_FPS) or 20.0
    W   = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
    H   = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap0.release()

    writer = cv2.VideoWriter(demo_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    SECONDS_PER_CLIP = 6

    for p in picks:
        exp_id = os.path.basename(p).split("_")[0]
        print(f"🎞️ Adding {exp_id}")
        title = np.zeros((H, W, 3), np.uint8)
        cv2.putText(title, f"{exp_id}: YOLO + Depth + Seam + A* (test)",
                    (30, int(H*0.55)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        for _ in range(int(fps*0.8)): writer.write(title)

        cap = cv2.VideoCapture(p)
        max_frames = int(fps * SECONDS_PER_CLIP)
        count = 0
        while count < max_frames:
            ret, fr = cap.read()
            if not ret: break
            writer.write(fr)
            count += 1
        cap.release()

    writer.release()
    print(f"✅ Demo merged → {demo_out}")


🎞️ Adding exp400
🎞️ Adding exp401
🎞️ Adding exp402
🎞️ Adding exp403
🎞️ Adding exp405
🎞️ Adding exp406
✅ Demo merged → C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\smoketest_from_outputs.mp4


In [17]:
# === TEST CELL: EXP700 + EXP701 QUICK TEST ===
import os, re, glob, cv2, numpy as np

PICK_IDS = [700, 701]           # 👈 your chosen experiments
FRAMES_PER_EXP = 150             # number of frames per video to process
TEST_OUT = os.path.join(VID_DIR, "test_700_701.mp4")

def _sort_numeric(paths):
    def keyfn(p):
        m = re.search(r"(\d+)(?=\.[^.]+$)", p)
        return int(m.group(1)) if m else 0
    return sorted(paths, key=keyfn)

# --- Find first valid frame for size ---
W = H = None
for e in PICK_IDS:
    exp_dir = rf"{BASE_CORRIDOR}\exp{e}"
    frames = _sort_numeric(glob.glob(rf"{exp_dir}\*.jpg") + 
                           glob.glob(rf"{exp_dir}\*.jpeg") + 
                           glob.glob(rf"{exp_dir}\*.png"))
    if frames:
        img = cv2.imread(frames[0])
        if img is not None:
            H, W = img.shape[:2]
            break
assert H is not None, "No valid frames found for 700 or 701"

fps = 20.0
writer = cv2.VideoWriter(TEST_OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
print(f"▶ Writing test video → {TEST_OUT}")

def path_hits_obstacle(grid, path, k=40):
    for r, c in path[:k]:
        if grid[r, c] == 1:
            return True
    return False

for e in PICK_IDS:
    exp_dir = rf"{BASE_CORRIDOR}\exp{e}"
    frames = _sort_numeric(glob.glob(rf"{exp_dir}\*.jpg") + 
                           glob.glob(rf"{exp_dir}\*.jpeg") + 
                           glob.glob(rf"{exp_dir}\*.png"))
    if not frames:
        print(f"⚠️ No frames in exp{e}")
        continue

    print(f"▶ Processing exp{e}: {len(frames)} frames")

    # Title card
    title = np.zeros((H, W, 3), np.uint8)
    cv2.putText(title, f"exp{e} — YOLO + Depth + Seam + A* Test",
                (30, int(H*0.55)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
    for _ in range(int(fps*0.8)): writer.write(title)

    last_path = []
    hold_counter = 0

    for i, fp in enumerate(frames[:FRAMES_PER_EXP], 1):
        frame = cv2.imread(fp)
        if frame is None: 
            continue

        # --- YOLO detections ---
        det_full = []
        res = yolo.predict(source=frame, conf=DET_CONF, verbose=False,
                           device=0 if device=="cuda" else None)[0]
        HH, WW = frame.shape[:2]; A = HH*WW
        if res.boxes is not None:
            xyxy = res.boxes.xyxy.cpu().numpy()
            cls  = res.boxes.cls.cpu().numpy() if res.boxes.cls is not None else np.zeros(len(xyxy))
            conf = res.boxes.conf.cpu().numpy() if res.boxes.conf is not None else np.ones(len(xyxy))
            for (x1,y1,x2,y2), c, s in zip(xyxy, cls, conf):
                area_frac = max(1.0, (x2-x1)*(y2-y1))/A
                if area_frac < MIN_AREA_FRAC or area_frac > MAX_AREA_FRAC: 
                    continue
                det_full.append((int(x1),int(y1),int(x2),int(y2), float(s), int(c)))

        boxes = [(x1,y1,x2,y2) for (x1,y1,x2,y2,_,_) in det_full]
        mask = boxes_to_mask(frame.shape, boxes, inflate=INFLATE_PIXELS)

        # Depth probe
        if USE_DEPTH_PROBE and (i % PROBE_EVERY == 0):
            mask = cv2.bitwise_or(mask, depth_probe_mask(frame))

        # Seam/white wall fallback
        try:
            seam_mask = seam_plane_mask(frame)
        except NameError:
            seam_mask = edge_plane_mask(frame)
        mask = cv2.bitwise_or(mask, seam_mask)

        # Occupancy + A*
        grid  = build_occupancy(mask, dilate_iter=2)
        start = (int(H*0.85), int(W*0.5))
        goal  = (int(H*0.15), int(W*0.5))
        if not last_path or (i % REPLAN_EVERY == 0):
            last_path = astar(grid, start, goal)

        # Stop logic
        bottom = grid[int(H*0.80):, int(W*0.40):int(W*0.60)]
        blocked = (bottom.mean() > STOP_BLOCKED_THRESH)
        if blocked or len(boxes)==0 or mask.mean() < 0.01*255:
            last_path = []

        # Draw visuals
        vis = frame.copy()
        if seam_mask.any():
            red_overlay = np.zeros_like(vis)
            red_overlay[:,:,2] = seam_mask
            vis = cv2.addWeighted(vis, 1.0, red_overlay, 0.35, 0)
        for (x1,y1,x2,y2) in boxes:
            cv2.rectangle(vis,(x1,y1),(x2,y2),(0,0,255),2)
        for (r,c) in last_path:
            cv2.circle(vis,(c,r),2,(0,255,0),-1)
        if blocked:
            cv2.putText(vis, "STOP (Detected)", (10,H-20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
        cv2.putText(vis, f"exp{e} {i}/{len(frames)} path:{len(last_path)} boxes:{len(boxes)}",
                    (10,25), cv2.FONT_HERSHEY_SIMPLEX, 0.7,(255,255,255),2)
        writer.write(vis)

writer.release()
print("✅ Test video created →", TEST_OUT)


▶ Writing test video → C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\test_700_701.mp4
▶ Processing exp700: 204 frames
▶ Processing exp701: 184 frames
✅ Test video created → C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\test_700_701.mp4


# NursingHome

In [18]:
import os, glob, cv2, random, shutil
from pathlib import Path

# SOURCE: class-per-folder dataset you showed
SRC_ROOT = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\MyNursingHome\MyNursingHome\MYNursingHome\MYNursingHome"

# DEST: YOLO detection-style dataset we'll create
DST_ROOT = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo"
IM_TRAIN = Path(DST_ROOT, "images", "train"); IM_TRAIN.mkdir(parents=True, exist_ok=True)
IM_VAL   = Path(DST_ROOT, "images", "val");   IM_VAL.mkdir(parents=True, exist_ok=True)
LB_TRAIN = Path(DST_ROOT, "labels", "train"); LB_TRAIN.mkdir(parents=True, exist_ok=True)
LB_VAL   = Path(DST_ROOT, "labels", "val");   LB_VAL.mkdir(parents=True, exist_ok=True)

# Build ordered class list from folder names (stable alphabetic)
CLASSES = sorted([d.name for d in Path(SRC_ROOT).iterdir() if d.is_dir()])
CLASS_TO_ID = {c:i for i,c in enumerate(CLASSES)}
print("Classes:", CLASS_TO_ID)


Classes: {'basket_bin': 0, 'bed': 1, 'bench': 2, 'cabinet': 3, 'call_bell': 4, 'cane_stick': 5, 'chair': 6, 'door': 7, 'electric_socket': 8, 'fan': 9, 'fire_extinguisher': 10, 'handrail': 11, 'human_being': 12, 'rack': 13, 'refrigerator': 14, 'shower': 15, 'sink': 16, 'sofa': 17, 'table': 18, 'television': 19, 'toilet_seat': 20, 'walker': 21, 'wardrobe': 22, 'water_dispencer': 23, 'wheelchair': 24}


In [19]:
def write_yolo_label(txt_path, cls_id, w, h):
    # centered box; keep 80% of min dimension to avoid full-image boxes
    box_w = 0.8
    box_h = 0.8
    # If image is very narrow/tall, relax a bit
    if w > h: 
        box_w = 0.8 * (h / w) + 0.1  # keep roughly square on very wide images
    else:
        box_h = 0.8 * (w / h) + 0.1
    box_w = max(0.4, min(0.95, box_w))
    box_h = max(0.4, min(0.95, box_h))
    x, y = 0.5, 0.5
    with open(txt_path, "w") as f:
        f.write(f"{cls_id} {x:.6f} {y:.6f} {box_w:.6f} {box_h:.6f}\n")

def collect_images(root_dir):
    exts = ("*.jpg","*.jpeg","*.png","*.bmp")
    files = []
    for e in exts:
        files += glob.glob(str(Path(root_dir, e)))
    return files

# Gather and split per-class (90/10)
random.seed(42)
all_pairs = []  # (img_path, cls_id)
for cls in CLASSES:
    imgs = collect_images(Path(SRC_ROOT, cls))
    imgs.sort()
    cid = CLASS_TO_ID[cls]
    for p in imgs:
        all_pairs.append((p, cid))

print("Total images found:", len(all_pairs))

# Simple global split (or do per-class splits if you prefer)
random.shuffle(all_pairs)
val_ratio = 0.1
val_n = max(1, int(len(all_pairs)*val_ratio))
val_set = set([p for p,_ in all_pairs[:val_n]])

copied = {"train":0, "val":0}
for img_path, cid in all_pairs:
    img = cv2.imread(img_path)
    if img is None: 
        continue
    h, w = img.shape[:2]
    stem = Path(img_path).stem

    split = "val" if img_path in val_set else "train"
    im_dst = (IM_VAL if split=="val" else IM_TRAIN) / f"{stem}.jpg"
    lb_dst = (LB_VAL if split=="val" else LB_TRAIN) / f"{stem}.txt"

    # write image (as .jpg) + weak YOLO label
    cv2.imwrite(str(im_dst), img)
    write_yolo_label(str(lb_dst), cid, w, h)
    copied[split]+=1

print("Copied:", copied)


Total images found: 37500
Copied: {'train': 33750, 'val': 3750}


In [20]:
from pathlib import Path
yaml_path = Path(DST_ROOT, "data.yaml")
yaml_text = f"""# auto-generated
path: {DST_ROOT.replace('\\','/')}
train: images/train
val: images/val
nc: {len(CLASSES)}
names: {CLASSES}
"""
yaml_path.write_text(yaml_text, encoding="utf-8")
print("Wrote:", yaml_path)
print(yaml_text)


Wrote: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\data.yaml
# auto-generated
path: C:/Users/ADMIN/Desktop/hospital_avoidance_demo/datasets/nursing_yolo
train: images/train
val: images/val
nc: 25
names: ['basket_bin', 'bed', 'bench', 'cabinet', 'call_bell', 'cane_stick', 'chair', 'door', 'electric_socket', 'fan', 'fire_extinguisher', 'handrail', 'human_being', 'rack', 'refrigerator', 'shower', 'sink', 'sofa', 'table', 'television', 'toilet_seat', 'walker', 'wardrobe', 'water_dispencer', 'wheelchair']



In [3]:
from ultralytics import YOLO
from pathlib import Path
import os, torch

# Your dataset root
DST_ROOT = Path(r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo")

# Keep logs local
os.environ["WANDB_DISABLED"] = "true"

# If no GPU, make sure we don't pretend there is one
if not torch.cuda.is_available():
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)  # remove '0' if set

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(Path(DST_ROOT, "data.yaml")),
    epochs=10,
    imgsz=320,
    batch=16,
    mosaic=0.0,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
    fliplr=0.0,
    device="cpu",            # ✅ force CPU
    workers=2,               # Windows/Jupyter: keep small to avoid dataloader issues
    cache=True,
    patience=3,
    plots=False
)
print("Best weights:", results.best)
BEST_WEIGHTS = results.best


Ultralytics 8.3.221  Python-3.13.4 torch-2.8.0+cpu CPU (Intel Core i5-8365U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

AttributeError: 'DetMetrics' object has no attribute 'best'. See valid attributes below.

Utility class for computing detection metrics such as precision, recall, and mean average precision (mAP).

Attributes:
    names (dict[int, str]): A dictionary of class names.
    box (Metric): An instance of the Metric class for storing detection results.
    speed (dict[str, float]): A dictionary for storing execution times of different parts of the detection process.
    task (str): The task type, set to 'detect'.
    stats (dict[str, list]): A dictionary containing lists for true positives, confidence scores, predicted classes, target classes, and target images.
    nt_per_class: Number of targets per class.
    nt_per_image: Number of targets per image.

Methods:
    update_stats: Update statistics by appending new values to existing stat collections.
    process: Process predicted results for object detection and update metrics.
    clear_stats: Clear the stored statistics.
    keys: Return

In [4]:
# 🔧 Fix: get best weights from the trainer instead of results
BEST_WEIGHTS = model.trainer.best
LAST_WEIGHTS = model.trainer.last

print("Best weights:", BEST_WEIGHTS)
print("Last weights:", LAST_WEIGHTS)


Best weights: C:\Users\ADMIN\hospital\runs\detect\train4\weights\best.pt
Last weights: C:\Users\ADMIN\hospital\runs\detect\train4\weights\last.pt


In [10]:
from ultralytics import YOLO
from pathlib import Path

test_model = YOLO(BEST_WEIGHTS)

# Pick 6 sample images from your dataset's validation or images folder
sample_imgs = list((DST_ROOT / "images" / "val").glob("*.jpg"))[:6]
# If your dataset doesn’t have val/, try 'train' instead
if not sample_imgs:
    sample_imgs = list((DST_ROOT / "images" / "train").glob("*.jpg"))[:6]

# Run predictions and save visualizations
for p in sample_imgs:
    test_model.predict(
        source=str(p),
        imgsz=640,
        conf=0.25,
        save=True,
        project=str(DST_ROOT / "pred_vis"),
        name="nursing",
        exist_ok=True
    )

print("✅ Saved predictions to:", Path(DST_ROOT, "pred_vis", "nursing"))



image 1/1 C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\00001.jpg: 640x640 (no detections), 164.7ms
Speed: 10.5ms preprocess, 164.7ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\nursing

image 1/1 C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\00002.jpg: 640x608 (no detections), 156.1ms
Speed: 9.3ms preprocess, 156.1ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 608)
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\nursing

image 1/1 C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\00003.jpg: 640x480 (no detections), 125.0ms
Speed: 7.8ms preprocess, 125.0ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 480)
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis

In [11]:
# === in your imports/config cell where you create YOLO ===
from ultralytics import YOLO
yolo = YOLO(BEST_WEIGHTS)  # or paste the absolute path if in a new session


In [12]:
ALLOWED_CLASSES = {
    # existing…
    "person","chair","bench","bed","couch","table","bottle",
    # nursing-specific you have in folders:
    "wheelchair","walker","wardrobe","cabinet","bed","sofa","sink","refrigerator",
    "handrail","door","table","basket_bin","television","water_dispenser"
}


In [13]:
# 1) Map ALLOWED_CLASSES (names) -> class IDs from the trained model
ALLOWED_CLASSES = {
    "person","chair","bench","bed","couch","table","bottle",
    "wheelchair","walker","wardrobe","cabinet","sofa","sink","refrigerator",
    "handrail","door","basket_bin","television","water_dispenser"
}

name_map = test_model.names  # dict: id -> name
names_to_ids = {n: i for i, n in name_map.items()}

allowed_ids = []
missing = []
for n in sorted(ALLOWED_CLASSES):
    if n in names_to_ids:
        allowed_ids.append(names_to_ids[n])
    else:
        missing.append(n)

print("Allowed IDs:", allowed_ids)
if missing:
    print("⚠️ These class names aren’t in your model:", missing)


Allowed IDs: [0, 1, 2, 3, 6, 7, 11, 14, 16, 17, 18, 19, 21, 22, 24]
⚠️ These class names aren’t in your model: ['bottle', 'couch', 'person', 'water_dispenser']


In [14]:
# 2) Pick 8 sample images from val/ (fallback to train/)
val_dir = DST_ROOT / "images" / "val"
train_dir = DST_ROOT / "images" / "train"

sample_imgs = list(val_dir.glob("*.jpg"))[:8] or list(val_dir.glob("*.png"))[:8]
if not sample_imgs:
    sample_imgs = list(train_dir.glob("*.jpg"))[:8] or list(train_dir.glob("*.png"))[:8]

# Run predictions, only for allowed classes
for p in sample_imgs:
    test_model.predict(
        source=str(p),
        imgsz=640,
        conf=0.25,
        classes=allowed_ids if allowed_ids else None,  # only allowed classes
        save=True,
        project=str(DST_ROOT / "pred_vis"),
        name="nursing",
        exist_ok=True
    )

print("✅ Saved predictions to:", Path(DST_ROOT, "pred_vis", "nursing"))



image 1/1 C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\00001.jpg: 640x640 (no detections), 149.9ms
Speed: 6.0ms preprocess, 149.9ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\nursing

image 1/1 C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\00002.jpg: 640x608 (no detections), 119.5ms
Speed: 8.6ms preprocess, 119.5ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 608)
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\nursing

image 1/1 C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\00003.jpg: 640x480 (no detections), 89.5ms
Speed: 2.6ms preprocess, 89.5ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 480)
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\nu

In [15]:
# 3) Quick mAP on the validation split, filtered to allowed classes
val_results = test_model.val(
    data=str(DST_ROOT / "data.yaml"),
    split="val",
    imgsz=640,
    conf=0.001,       # low conf to evaluate fully
    iou=0.6,          # NMS IoU
    classes=allowed_ids if allowed_ids else None,
    plots=True        # saves PR curves, confusion matrix, etc. in runs/val
)
print(val_results)    # prints summary mAP and metrics


Ultralytics 8.3.221  Python-3.13.4 torch-2.8.0+cpu CPU (Intel Core i5-8365U 1.60GHz)
val: Fast image access  (ping: 0.10.0 ms, read: 128.4118.5 MB/s, size: 85.4 KB)
val: Scanning C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\labels\val.cache... 1383 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1383/1383 905.7Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 87/87 0.5it/s 2:551.6ss
                   all       1383        803       0.23      0.225      0.213     0.0636
            basket_bin         50         50      0.215      0.126      0.159      0.043
                   bed         56         56      0.388      0.464      0.306     0.0823
                 bench         42         42      0.278      0.174      0.129     0.0332
               cabinet         53         53     0.0486      0.151     0.0705     0.0219
                 chair         66         66      0.343      0.152      

In [17]:
# 4) Count detections per class on those sample images
from collections import Counter

det_counts = Counter()
for p in sample_imgs:
    r = test_model.predict(
        source=str(p),
        imgsz=640,
        conf=0.25,
        classes=allowed_ids if allowed_ids else None,
        verbose=False
    )[0]
    if r.boxes is not None and len(r.boxes) > 0:
        for cls_id in r.boxes.cls.int().tolist():
            det_counts[name_map[int(cls_id)]] += 1

print("Detections in sample batch:")
for k, v in det_counts.most_common():
    print(f"{k:>18}: {v}")


Detections in sample batch:
              sofa: 1


In [18]:
r = test_model.predict(sample_imgs, imgsz=640, conf=0.25, verbose=False)
print("Frames:", len(r))
print("First frame classes:", r[0].names if len(r) else {})


Frames: 8
First frame classes: {0: 'basket_bin', 1: 'bed', 2: 'bench', 3: 'cabinet', 4: 'call_bell', 5: 'cane_stick', 6: 'chair', 7: 'door', 8: 'electric_socket', 9: 'fan', 10: 'fire_extinguisher', 11: 'handrail', 12: 'human_being', 13: 'rack', 14: 'refrigerator', 15: 'shower', 16: 'sink', 17: 'sofa', 18: 'table', 19: 'television', 20: 'toilet_seat', 21: 'walker', 22: 'wardrobe', 23: 'water_dispencer', 24: 'wheelchair'}


In [19]:
print(test_model.names)  # id -> name


{0: 'basket_bin', 1: 'bed', 2: 'bench', 3: 'cabinet', 4: 'call_bell', 5: 'cane_stick', 6: 'chair', 7: 'door', 8: 'electric_socket', 9: 'fan', 10: 'fire_extinguisher', 11: 'handrail', 12: 'human_being', 13: 'rack', 14: 'refrigerator', 15: 'shower', 16: 'sink', 17: 'sofa', 18: 'table', 19: 'television', 20: 'toilet_seat', 21: 'walker', 22: 'wardrobe', 23: 'water_dispencer', 24: 'wheelchair'}


In [20]:
val_results = test_model.val(data=str(DST_ROOT/"data.yaml"), split="val", imgsz=640, plots=True)
print(val_results)  # check mAP and per-class metrics


Ultralytics 8.3.221  Python-3.13.4 torch-2.8.0+cpu CPU (Intel Core i5-8365U 1.60GHz)
val: Fast image access  (ping: 0.10.0 ms, read: 213.6193.3 MB/s, size: 75.4 KB)
val: Scanning C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\labels\val.cache... 1383 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1383/1383 1.2Mit/s 0.0s0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 87/87 0.5it/s 2:451.3ss
                   all       1383       1383      0.246      0.274      0.231     0.0772
            basket_bin         50         50      0.194       0.14      0.164     0.0467
                   bed         56         56      0.338      0.464      0.304     0.0847
                 bench         42         42      0.276       0.19      0.135     0.0387
               cabinet         53         53     0.0512       0.17     0.0655     0.0217
             call_bell         58         58      0.242       0.27      

In [22]:
import yaml
from pathlib import Path

with open(DST_ROOT / "data.yaml", "r") as f:
    d = yaml.safe_load(f)

print("Train:", d.get("train"))
print("Val:", d.get("val"))
print("Classes:", d.get("names"))


Train: images/train
Val: images/val
Classes: ['basket_bin', 'bed', 'bench', 'cabinet', 'call_bell', 'cane_stick', 'chair', 'door', 'electric_socket', 'fan', 'fire_extinguisher', 'handrail', 'human_being', 'rack', 'refrigerator', 'shower', 'sink', 'sofa', 'table', 'television', 'toilet_seat', 'walker', 'wardrobe', 'water_dispencer', 'wheelchair']


In [24]:
val_results = test_model.val(
    data=str(DST_ROOT / "data.yaml"),
    split="val",     # assumes you have val split; else use split='test' if defined
    imgsz=640,
    iou=0.6,
    conf=0.001,      # low conf so eval isn’t pruned
    plots=True       # saves PR curves & confusion matrix
)
print(val_results)            # summary
print(val_results.results_dict)  # per-metric numbers (mAP, P, R, etc.)


Ultralytics 8.3.221  Python-3.13.4 torch-2.8.0+cpu CPU (Intel Core i5-8365U 1.60GHz)
val: Fast image access  (ping: 0.10.1 ms, read: 81.483.8 MB/s, size: 102.5 KB)
val: Scanning C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\labels\val.cache... 1383 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1383/1383 1.7Mit/s 0.0s0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 87/87 0.5it/s 2:531.4ss
                   all       1383       1383      0.274      0.262      0.243     0.0765
            basket_bin         50         50      0.215      0.126      0.159      0.043
                   bed         56         56      0.388      0.464      0.306     0.0823
                 bench         42         42      0.278      0.174      0.129     0.0332
               cabinet         53         53     0.0486      0.151     0.0705     0.0219
             call_bell         58         58      0.273      0.241      0

In [25]:
from pathlib import Path

video_dir = Path(r"C:\path\to\downloaded\hospital_test_videos")   # <- your folder
out_dir   = DST_ROOT / "pred_vids"

video_paths = list(video_dir.rglob("*.mp4")) + list(video_dir.rglob("*.avi")) + \
              list(video_dir.rglob("*.mov")) + list(video_dir.rglob("*.mkv"))

print(f"Found {len(video_paths)} videos")
for vp in video_paths:
    test_model.predict(
        source=str(vp),
        imgsz=640,
        conf=0.25,       # try 0.15–0.3 if needed
        save=True,
        project=str(out_dir),
        name="nursing",
        exist_ok=True
    )

print("Saved annotated videos to:", out_dir / "nursing")


Found 0 videos
Saved annotated videos to: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vids\nursing


In [27]:
from pathlib import Path

video_paths = [
    Path(r"C:\Users\ADMIN\Downloads\6129930-hd_1920_1080_30fps.mp4"),
    Path(r"C:\Users\ADMIN\Downloads\6130553-hd_1920_1080_30fps (1).mp4"),
    Path(r"C:\Users\ADMIN\Downloads\6130553-hd_1920_1080_30fps.mp4"),
]

out_dir = DST_ROOT / "pred_vids"


In [29]:
val_results = test_model.val(
    data=str(DST_ROOT / "data.yaml"),
    split="val",
    imgsz=640,
    conf=0.001,
    plots=True
)
print(val_results)


Ultralytics 8.3.221  Python-3.13.4 torch-2.8.0+cpu CPU (Intel Core i5-8365U 1.60GHz)
val: Fast image access  (ping: 0.40.1 ms, read: 25.314.1 MB/s, size: 24.4 KB)
val: Scanning C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\labels\val.cache... 1383 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1383/1383 1.6Mit/s 0.0s0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 87/87 0.4it/s 3:221.6ss
                   all       1383       1383      0.246      0.274      0.231     0.0772
            basket_bin         50         50      0.194       0.14      0.164     0.0467
                   bed         56         56      0.338      0.464      0.304     0.0847
                 bench         42         42      0.276       0.19      0.135     0.0387
               cabinet         53         53     0.0512       0.17     0.0655     0.0217
             call_bell         58         58      0.242       0.27      0.

In [ ]:
video = r"C:\Users\ADMIN\Downloads\6130553-hd_1920_1080_30fps.mp4"
for imgsz in (640, 960):
    for conf in (0.30, 0.20, 0.15, 0.10):
        print(f"imgsz={imgsz} conf={conf}")
        test_model.predict(
            source=video,
            imgsz=imgsz,
            conf=conf,
            iou=0.45,
            save=True,
            project=str(DST_ROOT / "pred_vids"),
            name=f"sweep_is{imgsz}_c{int(conf*100)}",
            exist_ok=True,
            classes=None,
            verbose=False
        )


imgsz=640 conf=0.3
WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vids\sweep_is640_c30
imgsz=640 conf=0.2
WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Result

In [5]:
# === Nursing-home image test (no video) ===
from ultralytics import YOLO
from pathlib import Path
from collections import Counter

# 1) Ensure we have a model
try:
    test_model
except NameError:
    test_model = YOLO(BEST_WEIGHTS)  # assumes BEST_WEIGHTS from your training cell

# 2) Pick a handful of images from val/ (fallback to train/)
val_dir = DST_ROOT / "images" / "val"
train_dir = DST_ROOT / "images" / "train"

sample_imgs = sorted(list(val_dir.glob("*.jpg")) + list(val_dir.glob("*.png")))[:12]
if not sample_imgs:
    sample_imgs = sorted(list(train_dir.glob("*.jpg")) + list(train_dir.glob("*.png")))[:12]
assert sample_imgs, "No images found in images/val or images/train."

print(f"Testing on {len(sample_imgs)} images")
out_dir = DST_ROOT / "pred_vis" / "nursing_images"

# 3) (Optional) No class filter first—see everything it predicts
#    If you want to restrict later, set classes=allowed_ids (after verifying names)
det_counts = Counter()

for p in sample_imgs:
    res = test_model.predict(
        source=str(p),
        imgsz=640,      # bump to 960 if your objects are small/far
        conf=0.20,      # a bit permissive to catch more detections
        iou=0.45,
        save=True,
        project=str(DST_ROOT / "pred_vis"),
        name="nursing_images",
        exist_ok=True,
        verbose=False
    )[0]

    if res.boxes is not None and len(res.boxes):
        for cls_id in res.boxes.cls.int().tolist():
            det_counts[test_model.names[int(cls_id)]] += 1

print("\n✅ Saved annotated images to:", out_dir)
print("\nDetections in sample batch:")
for k, v in det_counts.most_common():
    print(f"{k:>18}: {v}")

# 4) Quick model names print (helps align any ALLOWED_CLASSES later)
print("\nModel class names (id -> name):")
print(test_model.names)

# 5) (Optional) Proper metrics on your val split (precision/recall/mAP)
#    Comment out if you only want visuals.
val_results = test_model.val(
    data=str(DST_ROOT / "data.yaml"),
    split="val",
    imgsz=640,
    conf=0.001,
    plots=True
)
print("\nValidation summary:", val_results)  # shows P, R, mAP@.5, mAP@.5:.95


Testing on 12 images
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\nursing_images
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\nursing_images
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\nursing_images
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\nursing_images
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\nursing_images
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\nursing_images
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\nursing_images
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\nursing_images
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\nurs

In [7]:
import yaml
from pathlib import Path

yaml_path = Path(DST_ROOT) / "data.yaml"

with open(yaml_path, "r", encoding="utf-8") as f:
    d = yaml.safe_load(f)

print("🗂 Dataset file:", yaml_path)
print("Train images:", d.get("train"))
print("Val images:", d.get("val"))
print("\nClass names:")
for i, n in enumerate(d.get("names", [])):
    print(f"  {i}: {n}")

# Check what the trained model itself knows
print("\nModel class names from weights:", test_model.names)


🗂 Dataset file: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\data.yaml
Train images: images/train
Val images: images/val

Class names:
  0: basket_bin
  1: bed
  2: bench
  3: cabinet
  4: call_bell
  5: cane_stick
  6: chair
  7: door
  8: electric_socket
  9: fan
  10: fire_extinguisher
  11: handrail
  12: human_being
  13: rack
  14: refrigerator
  15: shower
  16: sink
  17: sofa
  18: table
  19: television
  20: toilet_seat
  21: walker
  22: wardrobe
  23: water_dispencer
  24: wheelchair

Model class names from weights: {0: 'basket_bin', 1: 'bed', 2: 'bench', 3: 'cabinet', 4: 'call_bell', 5: 'cane_stick', 6: 'chair', 7: 'door', 8: 'electric_socket', 9: 'fan', 10: 'fire_extinguisher', 11: 'handrail', 12: 'human_being', 13: 'rack', 14: 'refrigerator', 15: 'shower', 16: 'sink', 17: 'sofa', 18: 'table', 19: 'television', 20: 'toilet_seat', 21: 'walker', 22: 'wardrobe', 23: 'water_dispencer', 24: 'wheelchair'}


In [8]:
from pathlib import Path
from ultralytics import YOLO
from collections import Counter

# ensure we're using your trained weights
try:
    test_model
except NameError:
    test_model = YOLO(BEST_WEIGHTS)

val_dir = DST_ROOT / "images" / "val"
train_dir = DST_ROOT / "images" / "train"

# pick N images from val; if empty, fallback to train
N = 24
val_imgs = sorted(list(val_dir.glob("*.jpg")) + list(val_dir.glob("*.png")))[:N]
if not val_imgs:
    val_imgs = sorted(list(train_dir.glob("*.jpg")) + list(train_dir.glob("*.png")))[:N]

print(f"Testing on {len(val_imgs)} images from your dataset")

det_counts = Counter()
hits = 0
for p in val_imgs:
    r = test_model.predict(
        source=str(p),
        imgsz=960,        # helps small objects
        conf=0.15,        # permissive to surface candidates
        iou=0.45,
        classes=None,     # no filtering while debugging
        agnostic_nms=True,
        save=True,
        project=str(DST_ROOT / "pred_vis"),
        name="in_dataset_val",
        exist_ok=True,
        verbose=False
    )[0]
    if r.boxes is not None and len(r.boxes):
        hits += 1
        for cid in r.boxes.cls.int().tolist():
            det_counts[test_model.names[int(cid)]] += 1

print(f"\nImages with ≥1 detection: {hits}/{len(val_imgs)}")
print("Per-class detections:")
for k, v in det_counts.most_common():
    print(f"{k:>18}: {v}")

print("\nOverlays saved to:", DST_ROOT / "pred_vis" / "in_dataset_val")


Testing on 24 images from your dataset
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\in_dataset_val
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\in_dataset_val
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\in_dataset_val
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\in_dataset_val
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\in_dataset_val
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\in_dataset_val
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\in_dataset_val
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\pred_vis\in_dataset_val
Results saved to C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_

In [9]:
val_results = test_model.val(
    data=str(DST_ROOT / "data.yaml"),
    split="val",
    imgsz=640,
    conf=0.001,
    plots=True
)
print(val_results)  # shows P, R, mAP@.5, mAP@.5:.95


Ultralytics 8.3.221  Python-3.13.4 torch-2.8.0+cpu CPU (Intel Core i5-8365U 1.60GHz)
val: Fast image access  (ping: 0.40.1 ms, read: 46.121.1 MB/s, size: 53.9 KB)
val: Scanning C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\labels\val.cache... 1383 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1383/1383 911.1Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 87/87 0.4it/s 4:012.3ss
                   all       1383       1383      0.246      0.274      0.231     0.0772
            basket_bin         50         50      0.194       0.14      0.164     0.0467
                   bed         56         56      0.338      0.464      0.304     0.0847
                 bench         42         42      0.276       0.19      0.135     0.0387
               cabinet         53         53     0.0512       0.17     0.0655     0.0217
             call_bell         58         58      0.242       0.27      0.

In [4]:
from pathlib import Path
import yaml
from ultralytics import YOLO

# ---- Paths ----
RUN_DIR   = Path(r"C:\Users\ADMIN\hospital\runs\detect\train4")
WEIGHTS   = RUN_DIR / "weights" / "best.pt"
ROOT      = Path(r"C:\Users\ADMIN\hospital")  # your project root
ARGS_YAML = RUN_DIR / "args.yaml"

# ---- Find data.yaml (prefer previous run's setting) ----
DATA_YAML = None

# 1) Reuse what train4 used
if ARGS_YAML.exists():
    with open(ARGS_YAML, "r", encoding="utf-8") as f:
        args = yaml.safe_load(f) or {}
    prev_data = args.get("data")
    if prev_data:
        # Make path absolute if it was relative
        prev_data_path = Path(prev_data)
        DATA_YAML = prev_data_path if prev_data_path.is_absolute() else (ROOT / prev_data_path)

# 2) If not found, auto-search common places under ROOT
if DATA_YAML is None or not DATA_YAML.exists():
    candidates = list(ROOT.rglob("data*.yaml"))
    DATA_YAML = next((p for p in candidates if p.name.lower().startswith("data") and p.is_file()), None)

# 3) Final sanity check
if DATA_YAML is None or not DATA_YAML.exists():
    raise FileNotFoundError(
        "Couldn't locate data.yaml. "
        "Please point DATA_YAML to your dataset config file (it defines train/val paths and class names)."
    )

print("Using dataset yaml:", DATA_YAML)

# ---- Train ----
model = YOLO(WEIGHTS.as_posix())  # as_posix() avoids backslash issues
results = model.train(
    data=DATA_YAML.as_posix(),   # ✅ pass the correct dataset yaml
    epochs=10,
    imgsz=512,
    batch=8,
    device="cpu",
    workers=0,        # Windows/Jupyter
    cache=False,
    mosaic=0.5,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,
    fliplr=0.5,
    degrees=5, scale=0.7, shear=2, perspective=0.0005,
    patience=5,
    plots=False,
    project=str(ROOT / "runs"),
    name="fine_tune_10ep",
    exist_ok=True,
)

print("✅ Best weights after 10 epochs:", model.trainer.best)


Using dataset yaml: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\data.yaml
Ultralytics 8.3.221  Python-3.13.4 torch-2.8.0+cpu CPU (Intel Core i5-8365U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/Users/ADMIN/Desktop/hospital_avoidance_demo/datasets/nursing_yolo/data.yaml, degrees=5, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:/Users/ADMIN/hospital/runs/detect/train4/weights/best.

In [5]:
from ultralytics import YOLO
from pathlib import Path
from collections import Counter
import random
import time

# --- Paths ---
BEST = r"C:\Users\ADMIN\hospital\runs\fine_tune_10ep\weights\best.pt"
VAL_IMG_DIR = Path(r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val")
OUT_PROJECT = r"C:\Users\ADMIN\hospital\runs"  # Ultralytics will create a subfolder under this

# --- Pick 10 images deterministically (change seed if you want different ones) ---
random.seed(42)
img_paths = [p for p in VAL_IMG_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}]
assert len(img_paths) > 0, f"No images found in {VAL_IMG_DIR}"
if len(img_paths) < 10:
    print(f"Only found {len(img_paths)} images, using all of them.")
    test_imgs = img_paths
else:
    test_imgs = random.sample(img_paths, 10)

print("Testing these images:")
for p in test_imgs: 
    print(" -", p)

# --- Run prediction ---
model = YOLO(BEST)
stamp = time.strftime("%Y%m%d_%H%M%S")
run_name = f"test10_{stamp}"

results = model.predict(
    source=[str(p) for p in test_imgs],
    imgsz=640,
    conf=0.25,          # raise/lower if needed
    iou=0.5,
    device="cpu",
    save=True,          # save annotated images
    save_txt=True,      # save YOLO txts
    save_conf=True,     # write confidences in txts
    project=OUT_PROJECT,
    name=run_name,
    exist_ok=True,
    verbose=False
)

# --- Pretty print a compact summary ---
names = model.names
total_counts = Counter()

print("\nPer-image detections:")
for r in results:
    img_name = Path(r.path).name
    boxes = r.boxes
    if boxes is None or len(boxes) == 0:
        print(f" {img_name}: 0 detections")
        continue

    cls_ids = boxes.cls.tolist()
    confs = boxes.conf.tolist()
    per_img_counts = Counter(int(c) for c in cls_ids)
    total_counts.update(per_img_counts)

    # Show top few detections with conf
    topk = min(5, len(cls_ids))
    top_list = [f"{names[int(cls_ids[i])]} ({confs[i]:.2f})" for i in range(topk)]
    print(f" {img_name}: {sum(per_img_counts.values())} detections | top: {', '.join(top_list)}")

print("\nAggregate counts across 10 images:")
for cls_id, cnt in total_counts.most_common():
    print(f" - {names[cls_id]}: {cnt}")

print(f"\n✅ Saved annotated images & labels to: {Path(OUT_PROJECT) / run_name}")


Testing these images:
 - C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\01420.jpg
 - C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\00240.jpg
 - C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\00055.jpg
 - C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\00605.jpg
 - C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\00534.jpg
 - C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\00487.jpg
 - C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\00306.jpg
 - C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\00220.jpg
 - C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\01210.jpg
 - C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\images\val\00187.jpg
Results saved to C:\Users\ADMIN\hospital\runs\te

In [7]:
import cv2, os
from pathlib import Path

HOSPITAL_VIDEO = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\hospital_videos\3197638-hd_1920_1080_25fps.mp4"

# New dataset root for this clip
HOSP_DS = Path(r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\hospital_clip1")
IMG_DIR = HOSP_DS / "images" / "train"
LBL_DIR = HOSP_DS / "labels" / "train"
for d in [IMG_DIR, LBL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Extract 1 frame every N frames (tune this)
FRAME_STRIDE = 10   # every 10th frame ~ 2.5 fps if source is 25fps

cap = cv2.VideoCapture(HOSPITAL_VIDEO)
assert cap.isOpened(), f"Cannot open {HOSPITAL_VIDEO}"

idx = 0
saved = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if idx % FRAME_STRIDE == 0:
        out_path = IMG_DIR / f"hop_{idx:06d}.jpg"
        cv2.imwrite(str(out_path), frame)
        saved += 1
    idx += 1

cap.release()
print(f"✅ Extracted {saved} frames to: {IMG_DIR}")


✅ Extracted 26 frames to: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\hospital_clip1\images\train


In [8]:
from ultralytics import YOLO
from pathlib import Path

BEST_CORRIDOR = r"C:\Users\ADMIN\hospital\runs\fine_tune_10ep\weights\best.pt"  # your current best
IMG_DIR = Path(r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\hospital_clip1\images\train")
LBL_DIR = Path(r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\hospital_clip1\labels\train")
LBL_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO(BEST_CORRIDOR)
names = model.names

# Keep only classes relevant to avoidance
OBSTACLE_CLASSES = {
    'human_being','wheelchair','walker','bed','chair','table','sofa','rack',
    'wardrobe','refrigerator','television','sink','toilet_seat','bench','cabinet',
    'water_dispencer','fire_extinguisher','door','handrail','cane_stick','basket_bin','electric_socket','fan','shower'
}
allowed_ids = {cid for cid, cname in names.items() if cname in OBSTACLE_CLASSES}

# Inference & write YOLO txt labels: "cls cx cy w h" (normalized), no confidences
images = sorted([p for p in IMG_DIR.glob("*.jpg")])
results = model.predict(source=[str(p) for p in images], imgsz=640, conf=0.35, iou=0.5, device="cpu", verbose=False)

cnt = 0
for r in results:
    im_path = Path(r.path)
    h, w = r.orig_shape
    ypath = (LBL_DIR / im_path.stem).with_suffix(".txt")
    lines = []
    if r.boxes is not None and len(r.boxes) > 0:
        for cls, xywhn in zip(r.boxes.cls.tolist(), r.boxes.xywhn.tolist()):
            cid = int(cls)
            if cid not in allowed_ids:
                continue
            cx, cy, bw, bh = xywhn
            # clamp just in case
            cx = min(max(cx, 0.0), 1.0)
            cy = min(max(cy, 0.0), 1.0)
            bw = min(max(bw, 0.0), 1.0)
            bh = min(max(bh, 0.0), 1.0)
            lines.append(f"{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    with open(ypath, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    cnt += 1

print(f"✅ Wrote pseudo-labels for {cnt} images to: {LBL_DIR}")


✅ Wrote pseudo-labels for 26 images to: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\hospital_clip1\labels\train


In [9]:
from pathlib import Path
import yaml

ROOT = Path(r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo")
NURSING = ROOT / r"datasets\nursing_yolo"
HOSP_CLIP = ROOT / r"datasets\hospital_clip1"

COMBINED_YAML = ROOT / r"datasets\combined_hospital_corridor.yaml"

data = {
    "path": str(ROOT).replace("\\", "/"),
    "train": [
        str((NURSING / "images" / "train")).replace("\\","/"),
        str((HOSP_CLIP / "images" / "train")).replace("\\","/"),
    ],
    "val": str((NURSING / "images" / "val")).replace("\\","/"),
    "nc": 25,
    "names": {
        0: 'basket_bin', 1: 'bed', 2: 'bench', 3: 'cabinet', 4: 'call_bell', 5: 'cane_stick',
        6: 'chair', 7: 'door', 8: 'electric_socket', 9: 'fan', 10: 'fire_extinguisher',
        11: 'handrail', 12: 'human_being', 13: 'rack', 14: 'refrigerator', 15: 'shower',
        16: 'sink', 17: 'sofa', 18: 'table', 19: 'television', 20: 'toilet_seat',
        21: 'walker', 22: 'wardrobe', 23: 'water_dispencer', 24: 'wheelchair'
    }
}

with open(COMBINED_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("✅ Wrote:", COMBINED_YAML)


✅ Wrote: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\combined_hospital_corridor.yaml


In [10]:
from ultralytics import YOLO
from pathlib import Path

BEST_START = r"C:\Users\ADMIN\hospital\runs\fine_tune_10ep\weights\best.pt"  # start from your good checkpoint
COMBINED_YAML = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\combined_hospital_corridor.yaml"

model = YOLO(BEST_START)
results = model.train(
    data=COMBINED_YAML,
    epochs=10,        # start small; increase to 20–30 if you like
    imgsz=512,
    batch=8,
    device="cpu",
    workers=0,
    cache=False,
    mosaic=0.5,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,
    fliplr=0.5,
    degrees=5, scale=0.7, shear=2, perspective=0.0005,
    patience=5,
    plots=False,
    project=r"C:\Users\ADMIN\hospital\runs",
    name="unified_hosp_corridor_10ep",
    exist_ok=True,
)

print("✅ New BEST:", model.trainer.best)


Ultralytics 8.3.221  Python-3.13.4 torch-2.8.0+cpu CPU (Intel Core i5-8365U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\combined_hospital_corridor.yaml, degrees=5, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\ADMIN\hospital\runs\fine_tune_10ep\weights\best.pt, momentum=0.937, mosaic=0.5, multi_scale=False, name=unified_hosp_corridor_10ep, nbs=6

In [11]:
UNIFIED_BEST = r"C:\Users\ADMIN\hospital\runs\unified_hosp_corridor_10ep\weights\best.pt"

# in your avoidance code, replace:
# model = YOLO(BEST)
# with:
from ultralytics import YOLO
model = YOLO(UNIFIED_BEST)

# everything else (sectors / steering / CSV export) stays the same


In [12]:
from ultralytics import YOLO
import cv2, time
from pathlib import Path

UNIFIED_BEST = r"C:\Users\ADMIN\hospital\runs\unified_hosp_corridor_10ep\weights\best.pt"
HOSPITAL_VIDEO = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\hospital_videos\3197638-hd_1920_1080_25fps.mp4"

model = YOLO(UNIFIED_BEST)
cap = cv2.VideoCapture(HOSPITAL_VIDEO)
assert cap.isOpened()

N_FRAMES, STEP, shown = 20, 5, 0
i = 0
while shown < N_FRAMES:
    ok, frame = cap.read()
    if not ok: break
    if i % STEP == 0:
        r = model.predict(source=frame, imgsz=640, conf=0.35, iou=0.5, device="cpu", verbose=False)[0]
        im = r.plot()  # annotated frame
        out = f"C:/Users/ADMIN/hospital/runs/smoke_{shown:02d}.jpg"
        cv2.imwrite(out, im)
        print("saved:", out)
        shown += 1
    i += 1

cap.release()
print("✅ Smoke test frames saved under C:\\Users\\ADMIN\\hospital\\runs")


saved: C:/Users/ADMIN/hospital/runs/smoke_00.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_01.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_02.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_03.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_04.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_05.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_06.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_07.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_08.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_09.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_10.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_11.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_12.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_13.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_14.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_15.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_16.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_17.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_18.jpg
saved: C:/Users/ADMIN/hospital/runs/smoke_19.jpg
✅ Smoke test frames 

In [1]:
# ------ knobs for fast test ------
UNIFIED_BEST = r"C:\Users\ADMIN\hospital\runs\unified_hosp_corridor_10ep\weights\best.pt"
VIDEO_DIR     = Path(r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\videos_in")  # put your hospital video here (or point to folder)
ONLY_THIS     = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\hospital_videos\3197638-hd_1920_1080_25fps.mp4"  # optional single file
MAX_SECONDS   = 20         # stop early
SKIP          = 2          # process every 3rd frame
IMGSZ         = 416        # faster than 640
CONF, IOU     = 0.30, 0.5
DEVICE        = "cpu"      # change to "cuda" if available
# ---------------------------------


NameError: name 'Path' is not defined

In [2]:
from ultralytics import YOLO
import cv2
import numpy as np
from pathlib import Path
from collections import Counter
import time, os

# --- Config ---
UNIFIED_BEST = r"C:\Users\ADMIN\hospital\runs\unified_hosp_corridor_10ep\weights\best.pt"
HOSPITAL_VIDEO = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\hospital_videos\3197638-hd_1920_1080_25fps.mp4"
OUT_DIR = Path(r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Model & avoidance params
IMGSZ = 416
CONF = 0.3
IOU = 0.5
DEVICE = "cpu"        # change to "cuda" if GPU available
SECTORS = 9           # horizontal divisions
MAX_STEER_DEG = 30.0  # ± degrees
BLOCK_W_FRAC = 0.18   # consider blocked if box > 18% width
BLOCK_AREA_FRAC = 0.06

# --- Load Model ---
model = YOLO(UNIFIED_BEST)
names = model.names
OBSTACLE_CLASSES = {
    'human_being','wheelchair','walker','bed','chair','table','sofa','rack','wardrobe',
    'refrigerator','television','sink','toilet_seat','bench','cabinet','water_dispencer',
    'fire_extinguisher','door','handrail','cane_stick','basket_bin','electric_socket','fan','shower'
}
obstacle_ids = {cid for cid, cname in names.items() if cname in OBSTACLE_CLASSES}

# --- Helper functions ---
def choose_sector(grid_1xK):
    k = int(np.argmin(grid_1xK))
    center = (len(grid_1xK)-1) / 2.0
    angle = (k - center) / center * MAX_STEER_DEG
    return k, float(angle)

def sector_index(x_center, width_px):
    sec_w = width_px / SECTORS
    k = int(np.clip(x_center // sec_w, 0, SECTORS-1))
    return k

def draw_grid_panel(sector_scores, h=400, w=400, best_k=None):
    panel = np.zeros((h, w, 3), dtype=np.uint8)
    sec_w = w // SECTORS
    for k in range(SECTORS):
        x0 = k * sec_w
        x1 = w if k == SECTORS-1 else (k+1) * sec_w
        val = int(np.clip(sector_scores[k] * 255, 0, 255))
        panel[:, x0:x1, :] = (val, val, val)
        color = (0, 0, 255) if k == best_k else (80, 80, 80)
        cv2.rectangle(panel, (x0, 0), (x1-1, h-1), color, 2)
        cv2.putText(panel, str(k), (x0+6, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1, cv2.LINE_AA)
    return panel

# --- Process Video ---
cap = cv2.VideoCapture(HOSPITAL_VIDEO)
assert cap.isOpened(), "❌ Could not open hospital video"

fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Output writer
stamp = time.strftime("%Y%m%d_%H%M%S")
out_path = OUT_DIR / f"hospital_test_{stamp}.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
panel = draw_grid_panel(np.zeros(SECTORS), h=H, w=max(W//3, 360))
out_w = W + panel.shape[1]
out_h = H
writer = cv2.VideoWriter(str(out_path), fourcc, fps, (out_w, out_h))

print(f"▶️ Processing: {HOSPITAL_VIDEO}")
frame_i = 0
steer_ema = 0.0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    # Predict
    res = model.predict(frame, imgsz=IMGSZ, conf=CONF, iou=IOU, device=DEVICE, verbose=False)[0]
    sector_scores = np.zeros(SECTORS, dtype=np.float32)
    h, w = frame.shape[:2]
    img_area = float(h*w)

    # Fill sector scores
    if res.boxes is not None and len(res.boxes) > 0:
        for i in range(len(res.boxes)):
            cls_id = int(res.boxes.cls[i].item())
            conf = float(res.boxes.conf[i].item())
            x1, y1, x2, y2 = res.boxes.xyxy[i].tolist()
            cname = names.get(cls_id, str(cls_id))
            if cls_id in obstacle_ids:
                bw = (x2 - x1)
                bh = (y2 - y1)
                area = (bw * bh) / img_area
                x_center = 0.5 * (x1 + x2)
                k = sector_index(x_center, w)
                if (bw / w) > BLOCK_W_FRAC or area > BLOCK_AREA_FRAC:
                    sector_scores[k] = 1.0
            # Draw detection
            color = (0, 255, 0) if cls_id in obstacle_ids else (255, 255, 0)
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            cv2.putText(frame, f"{cname} {conf:.2f}", (int(x1), max(0, int(y1)-6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 1, cv2.LINE_AA)

    # Steering
    best_k, steer_deg = choose_sector(sector_scores)
    alpha = 0.2
    steer_ema = alpha * steer_deg + (1 - alpha) * steer_ema
    panel = draw_grid_panel(sector_scores, h=H, w=max(W//3, 360), best_k=best_k)

    # Overlay steering text
    cv2.putText(frame, f"STEER: {steer_ema:+.1f} deg", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (30,144,255), 2, cv2.LINE_AA)

    combined = np.concatenate([frame, panel], axis=1)
    writer.write(combined)
    frame_i += 1

cap.release()
writer.release()
print(f"✅ Saved avoidance video to:\n{out_path}")


▶️ Processing: C:\Users\ADMIN\Desktop\hospital_avoidance_demo\hospital_videos\3197638-hd_1920_1080_25fps.mp4
✅ Saved avoidance video to:
C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\hospital_test_20251027_084832.mp4


In [1]:
from ultralytics import YOLO
import cv2, numpy as np, time
from pathlib import Path

# --- Paths ---
UNIFIED_BEST   = r"C:\Users\ADMIN\hospital\runs\unified_hosp_corridor_10ep\weights\best.pt"
HOSPITAL_VIDEO = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\hospital_videos\3197638-hd_1920_1080_25fps.mp4"
OUT_DIR        = Path(r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Inference knobs (no retraining) ---
IMGSZ, CONF, IOU = 416, 0.30, 0.50
DEVICE = "cpu"               # "cuda" if you have GPU
TRACKER_YAML = "bytetrack.yaml"
BOX_EMA = 0.6                # 0..1 (higher = smoother, lower = more responsive)
TRACK_TTL = 45               # keep a track this many frames after it disappears (hysteresis)

# --- Avoidance knobs ---
SECTORS = 9
MAX_STEER_DEG = 30.0
BLOCK_W_FRAC = 0.18
BLOCK_AREA_FRAC = 0.06

# --- Load model & classes ---
model = YOLO(UNIFIED_BEST)
names = model.names
OBSTACLE_CLASSES = {
    'human_being','wheelchair','walker','bed','chair','table','sofa','rack','wardrobe',
    'refrigerator','television','sink','toilet_seat','bench','cabinet','water_dispencer',
    'fire_extinguisher','door','handrail','cane_stick','basket_bin','electric_socket','fan','shower'
}
obstacle_ids = {cid for cid, cname in names.items() if cname in OBSTACLE_CLASSES}

# --- Utils ---
def sector_index(x_center, width_px):
    sec_w = width_px / SECTORS
    return int(np.clip(x_center // sec_w, 0, SECTORS-1))

def choose_sector(grid_1xK):
    k = int(np.argmin(grid_1xK))
    center = (len(grid_1xK)-1) / 2.0
    angle = (k - center) / center * MAX_STEER_DEG
    return k, float(angle)

def draw_grid_panel(sector_scores, h, w, best_k=None):
    panel = np.zeros((h, w, 3), dtype=np.uint8)
    sec_w = w // SECTORS
    for k in range(SECTORS):
        x0 = k * sec_w
        x1 = w if k == SECTORS-1 else (k+1) * sec_w
        val = int(np.clip(sector_scores[k] * 255, 0, 255))
        panel[:, x0:x1, :] = (val, val, val)
        color = (0, 0, 255) if k == best_k else (80, 80, 80)
        cv2.rectangle(panel, (x0, 0), (x1-1, h-1), color, 2)
        cv2.putText(panel, str(k), (x0+6, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1, cv2.LINE_AA)
    return panel

def color_for_track(tid):
    rng = np.random.default_rng(int(tid) + 12345)
    return tuple(int(c) for c in rng.integers(60, 255, size=3))

# --- Tracking state (for smoothing) ---
tracks = {}  # tid -> {'xyxy': np.array([x1,y1,x2,y2]), 'last_seen': int, 'name': str, 'color': (b,g,r)}

# --- Open video ---
cap = cv2.VideoCapture(HOSPITAL_VIDEO)
assert cap.isOpened(), f"Could not open {HOSPITAL_VIDEO}"
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

panel_w = max(W//3, 360)
panel = draw_grid_panel(np.zeros(SECTORS), h=H, w=panel_w)
out_w, out_h = W + panel_w, H
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
stamp = time.strftime("%Y%m%d_%H%M%S")
out_path = OUT_DIR / f"video_anti_flicker_{stamp}.mp4"
writer = cv2.VideoWriter(str(out_path), fourcc, fps, (out_w, out_h))

print("▶️ Running anti-flicker hospital test...")
frame_i, steer_ema = 0, 0.0

while True:
    ok, frame = cap.read()
    if not ok: break
    h, w = frame.shape[:2]
    img_area = float(h*w)

    # Track (stable IDs)
    r = model.track(
        frame, imgsz=IMGSZ, conf=CONF, iou=IOU, device=DEVICE,
        tracker=TRACKER_YAML, persist=True, verbose=False
    )[0]

    # Sector scores (0 clear .. 1 blocked)
    sector_scores = np.zeros(SECTORS, dtype=np.float32)

    now = frame_i
    seen = set()

    if r.boxes is not None and len(r.boxes) > 0:
        boxes = r.boxes.xyxy.cpu().numpy()
        clses = r.boxes.cls.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()
        ids   = r.boxes.id
        ids = ids.cpu().numpy().astype(int) if ids is not None else np.arange(len(boxes))

        for (x1,y1,x2,y2), cid, conf, tid in zip(boxes, clses, confs, ids):
            cname = names.get(int(cid), str(cid))
            new_box = np.array([x1,y1,x2,y2], dtype=np.float32)

            # EMA smoothing per track
            if tid in tracks:
                tracks[tid]['xyxy'] = BOX_EMA * tracks[tid]['xyxy'] + (1 - BOX_EMA) * new_box
            else:
                tracks[tid] = {
                    'xyxy': new_box,
                    'last_seen': now,
                    'name': cname,
                    'color': color_for_track(tid)
                }
            tracks[tid]['last_seen'] = now
            seen.add(tid)

            sx1, sy1, sx2, sy2 = tracks[tid]['xyxy']
            color = tracks[tid]['color']

            # Avoidance scoring uses smoothed boxes (reduces flicker in logic too)
            if cid in obstacle_ids:
                bw, bh = (sx2 - sx1), (sy2 - sy1)
                area = (bw * bh) / img_area
                x_center = 0.5 * (sx1 + sx2)
                k = sector_index(x_center, w)
                if (bw / w) > BLOCK_W_FRAC or area > BLOCK_AREA_FRAC:
                    sector_scores[k] = 1.0

            # Draw smoothed box
            cv2.rectangle(frame, (int(sx1), int(sy1)), (int(sx2), int(sy2)), color, 2)
            cv2.putText(frame, f"{cname} {conf:.2f} id:{int(tid)}",
                        (int(sx1), max(0, int(sy1)-6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

    # Hysteresis: keep recently-lost tracks visible (for BOX_EMA stability)
    for tid, t in list(tracks.items()):
        if tid not in seen and (now - t['last_seen']) <= TRACK_TTL:
            sx1, sy1, sx2, sy2 = t['xyxy']
            cv2.rectangle(frame, (int(sx1), int(sy1)), (int(sx2), int(sy2)), t['color'], 2)
            cv2.putText(frame, f"{t['name']} (hold)",
                        (int(sx1), max(0, int(sy1)-6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, t['color'], 2, cv2.LINE_AA)
        elif (now - t['last_seen']) > TRACK_TTL:
            tracks.pop(tid, None)

    # Steering + indicators
    best_k, steer_deg = choose_sector(sector_scores)
    steer_ema = 0.2 * steer_deg + 0.8 * steer_ema  # smooth steering
    panel = draw_grid_panel(sector_scores, h=H, w=panel_w, best_k=best_k)

    center_k = SECTORS // 2
    forward_clear = (sector_scores[center_k] < 0.5) and (abs(steer_ema) < 8.0)
    floor_y, mid_x = int(h * 0.95), w // 2
    if forward_clear:
        # green corridor
        spread, top_y = int(w * 0.18), int(h * 0.65)
        pts = np.array([[mid_x-spread, floor_y],[mid_x+spread, floor_y],
                        [mid_x+spread//2, top_y],[mid_x-spread//2, top_y]], np.int32)
        cv2.fillPoly(frame, [pts], (60,200,60))
        cv2.circle(frame, (mid_x, floor_y), 12, (0,255,0), -1)
        cv2.putText(frame, "GO", (mid_x+18, floor_y+6), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2, cv2.LINE_AA)
    else:
        cv2.circle(frame, (mid_x, floor_y), 12, (0,0,255), -1)
        cv2.putText(frame, "STOP", (mid_x+18, floor_y+6), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2, cv2.LINE_AA)
        # arrow suggesting turn
        arrow_len = int(w * 0.12)
        dx = int(np.clip(steer_ema / MAX_STEER_DEG, -1, 1) * arrow_len)
        cv2.arrowedLine(frame, (mid_x, floor_y-20), (mid_x+dx, floor_y-20), (0,200,255), 4, tipLength=0.35)

    cv2.putText(frame, f"STEER: {steer_ema:+.1f} deg", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (30,144,255), 2, cv2.LINE_AA)

    combined = np.concatenate([frame, panel], axis=1)
    writer.write(combined)
    frame_i += 1

cap.release()
writer.release()
print(f"✅ Saved anti-flicker video to:\n{out_path}")


▶️ Running anti-flicker hospital test...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 1.3 MB/s eta 0:00:01
   --------------------- ------------------ 0.8/1.5 MB 1.3 MB/s eta 0:00:01
   ---------------------------- ----------- 1.0/1.5 MB 1.3 MB/s eta 0:00:01
   ---------------------------- ----------- 1.0/1.5 MB 1.3 MB/s eta 0:00:01
   ----------------------------------- ---- 1.3/1.5 MB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.1 MB/s  0:00:01

requirements: AutoUpdate success  12.6s
WARNING requirements: Restart runtime or rerun command for updates to take effect

✅ Saved anti-flicker video to:
C:\Users\ADMIN\Desktop\hospital_avoidance_dem

In [2]:
from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import numpy as np

# --- Paths (adjust if needed) ---
WEIGHTS   = r"C:\Users\ADMIN\hospital\runs\fine_tune_10ep\weights\best.pt"
DATA_YAML = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\data.yaml"
OUT_CSV   = Path(r"C:\Users\ADMIN\hospital\runs\metrics_val_per_class.csv")

# --- Eval on VAL split (set split='test' if your data.yaml defines one) ---
model = YOLO(WEIGHTS)
val_results = model.val(
    data=DATA_YAML,
    split="val",      # change to "test" if you have a test set defined
    imgsz=640,
    conf=0.001,
    iou=0.6,
    device="cpu",
    workers=0,
    plots=False,
    verbose=True
)

# --- Overall metrics ---
overall = val_results.results_dict
print("\n=== OVERALL (VAL) ===")
print(f"Precision: {overall['metrics/precision(B)']:.4f}")
print(f"Recall   : {overall['metrics/recall(B)']:.4f}")
print(f"mAP@0.50 : {overall['metrics/mAP50(B)']:.4f}")
print(f"mAP@0.50:0.95 : {overall['metrics/mAP50-95(B)']:.4f}")

# --- Per-class metrics to CSV ---
names = val_results.names                # {class_id: name}
maps  = val_results.maps                 # per-class mAP@0.50:0.95 (array)
ap50  = val_results.box.ap50             # per-class mAP@0.50 (array)
prec  = val_results.box.p                # precision curve summary per class (array)
rec   = val_results.box.r                # recall curve summary per class (array)

# Some Ultralytics builds expose per-class p/r directly; if not, derive from curves:
# Fallback if p/r are None (handle gracefully)
if prec is None or isinstance(prec, (float, np.floating)):
    prec = np.full_like(maps, np.nan, dtype=float)
if rec is None or isinstance(rec, (float, np.floating)):
    rec = np.full_like(maps, np.nan, dtype=float)

df = pd.DataFrame({
    "class_id": list(names.keys()),
    "class_name": [names[i] for i in names.keys()],
    "precision": prec,
    "recall": rec,
    "mAP50": ap50,
    "mAP50_95": maps
}).sort_values("class_id").reset_index(drop=True)

df.to_csv(OUT_CSV, index=False)
print(f"\n📄 Per-class metrics saved to: {OUT_CSV}")


Ultralytics 8.3.221  Python-3.13.4 torch-2.8.0+cpu CPU (Intel Core i5-8365U 1.60GHz)
Model summary (fused): 72 layers, 3,010,523 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.40.1 ms, read: 69.921.7 MB/s, size: 80.3 KB)
val: Scanning C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\labels\val.cache... 1383 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1383/1383 947.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 87/87 0.5it/s 2:571.5ss
                   all       1383       1383      0.843      0.836      0.878      0.662
            basket_bin         50         50      0.802       0.92      0.907      0.712
                   bed         56         56       0.95      0.804      0.903      0.633
                 bench         42         42      0.803      0.833      0.859      0.627
               cabinet         53         53      0.512      0.755      0.626      

In [4]:
# === CELL: Inference Booster (TTA + WBF + Soft-NMS + Hysteresis) ===
import numpy as np
import cv2
from ultralytics import YOLO

# ---- knobs you can tweak (sane defaults) ----
DET_BASE_CONF       = 0.38   # base detector conf for raw YOLO calls
SHOW_THRESH         = 0.45   # show a box if conf >= this
DROP_THRESH         = 0.28   # remove a tracked box only when conf < this (hysteresis)
NMS_IOU             = 0.55   # NMS/WBF IoU threshold (0.5–0.65 good)
SOFT_NMS_SIGMA      = 0.5    # softer suppression (0.4–0.7); set 0 for hard NMS
WBF_IOU             = 0.55   # merge threshold for fusion
WBF_SKIP_CONF       = 0.10   # ignore ultra-low conf preds before fusion
PRIORITY_CLASSES    = set()  # e.g. {"door","cabinet"} — optional per-class boost

# (optional) small boost to priority classes after fusion
PRIORITY_BOOST      = 0.05   # +0.05 conf

# ---------- helpers ----------
def _preprocess_for_yolo(img):
    """CLAHE + mild gamma—robust to hospital lighting."""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    img2 = cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2BGR)
    # mild gamma
    g = 0.9
    img2 = np.clip((img2/255.0)**g, 0, 1.0)
    return (img2*255).astype(np.uint8)

def _soft_nms(boxes, scores, iou_thr=NMS_IOU, sigma=SOFT_NMS_SIGMA, score_thr=0.001):
    """Gaussian Soft-NMS on (x1,y1,x2,y2) + scores; returns filtered indices."""
    boxes = boxes.copy().astype(np.float32)
    scores = scores.copy().astype(np.float32)
    N = len(scores)
    idxs = np.arange(N)
    keep = []

    while idxs.size > 0:
        i = idxs[np.argmax(scores[idxs])]
        keep.append(i)
        idxs = idxs[idxs != i]
        if idxs.size == 0: break

        xx1 = np.maximum(boxes[i,0], boxes[idxs,0])
        yy1 = np.maximum(boxes[i,1], boxes[idxs,1])
        xx2 = np.minimum(boxes[i,2], boxes[idxs,2])
        yy2 = np.minimum(boxes[i,3], boxes[idxs,3])

        w = np.maximum(0.0, xx2-xx1)
        h = np.maximum(0.0, yy2-yy1)
        inter = w*h
        area_i = (boxes[i,2]-boxes[i,0])*(boxes[i,3]-boxes[i,1])
        area_j = (boxes[idxs,2]-boxes[idxs,0])*(boxes[idxs,3]-boxes[idxs,1])
        iou = inter / (area_i + area_j - inter + 1e-9)

        if sigma > 0:
            # Gaussian decay
            decay = np.exp(-(iou**2)/sigma)
            scores[idxs] *= decay
        else:
            # hard NMS behavior
            scores[idxs][iou > iou_thr] = 0.0

        idxs = idxs[scores[idxs] > score_thr]
    return np.array(keep, dtype=int)

def _wbf(box_sets, score_sets, label_sets, iou_thr=WBF_IOU, skip_conf=WBF_SKIP_CONF):
    """
    Weighted Boxes Fusion across TTA sets.
    Inputs are lists over TTA of arrays: boxes Nx4, scores Nx1, labels Nx1.
    Returns fused (boxes, scores, labels).
    """
    boxes_all, scores_all, labels_all = [], [], []
    for b, s, l in zip(box_sets, score_sets, label_sets):
        if b.size == 0: continue
        m = s >= skip_conf
        boxes_all.append(b[m])
        scores_all.append(s[m])
        labels_all.append(l[m])
    if not boxes_all:
        return np.zeros((0,4), np.float32), np.zeros((0,), np.float32), np.zeros((0,), np.int32)

    boxes = np.concatenate(boxes_all, 0)
    scores = np.concatenate(scores_all, 0)
    labels = np.concatenate(labels_all, 0)

    # class-wise fusion
    out_boxes, out_scores, out_labels = [], [], []
    classes = np.unique(labels)
    for c in classes:
        m = labels == c
        B = boxes[m]; S = scores[m]
        if B.size == 0: continue
        picked = []
        used = np.zeros(len(S), dtype=bool)
        for i in np.argsort(-S):
            if used[i]: continue
            cluster = [i]
            used[i] = True
            # merge overlapping with IoU >= iou_thr
            for j in np.where(~used)[0]:
                # IoU
                xx1 = max(B[i,0], B[j,0]); yy1 = max(B[i,1], B[j,1])
                xx2 = min(B[i,2], B[j,2]); yy2 = min(B[i,3], B[j,3])
                w = max(0.0, xx2-xx1); h = max(0.0, yy2-yy1)
                inter = w*h
                iou = inter / ((B[i,2]-B[i,0])*(B[i,3]-B[i,1]) + (B[j,2]-B[j,0])*(B[j,3]-B[j,1]) - inter + 1e-9)
                if iou >= iou_thr:
                    used[j] = True
                    cluster.append(j)
            # weighted merge
            ww = S[cluster]
            bb = B[cluster]
            wsum = np.sum(ww) + 1e-9
            fused = np.sum(bb * ww[:,None], axis=0) / wsum
            score = np.max(ww)  # or np.mean(ww)
            out_boxes.append(fused); out_scores.append(score); out_labels.append(int(c))

    out_boxes  = np.array(out_boxes,  dtype=np.float32)
    out_scores = np.array(out_scores, dtype=np.float32)
    out_labels = np.array(out_labels, dtype=np.int32)
    return out_boxes, out_scores, out_labels

def yolo_obstacles_boosted(frame_bgr, model: YOLO, allowed_classes=None):
    """
    Drop-in replacement for yolo_obstacles():
      - Preprocess
      - TTA (orig + hflip + 0.9/1.1 scale)
      - WBF fusion
      - Soft-NMS
      - Hysteresis via SHOW_THRESH/DROP_THRESH (you still can use BoxMemory on top)
    Returns: list of (x1,y1,x2,y2, conf, cname)
    """
    H, W = frame_bgr.shape[:2]
    names = getattr(model.model, "names", None)

    # --- Preprocess base image
    base = _preprocess_for_yolo(frame_bgr)

    # --- Build TTA views and inverse transforms
    views = []
    invs  = []

    # 1) original
    views.append(base)
    invs.append(lambda boxes: boxes)

    # 2) horizontal flip
    views.append(cv2.flip(base, 1))
    invs.append(lambda b: np.stack([W - b[:,2], b[:,1], W - b[:,0], b[:,3]], axis=1) if b.size else b)

    # 3) scale 0.9
    s = 0.9
    v = cv2.resize(base, (int(W*s), int(H*s)))
    pad = np.zeros_like(base); pad[:v.shape[0], :v.shape[1]] = v
    views.append(pad)
    def inv_scale09(b):
        if b.size == 0: return b
        return np.stack([b[:,0]/s, b[:,1]/s, b[:,2]/s, b[:,3]/s], axis=1)
    invs.append(inv_scale09)

    # 4) scale 1.1 (center crop back)
    s = 1.1
    big = cv2.resize(base, (int(W*s), int(H*s)))
    offx = (big.shape[1]-W)//2; offy = (big.shape[0]-H)//2
    crop = big[offy:offy+H, offx:offx+W]
    views.append(crop)
    def inv_scale11(b):
        if b.size == 0: return b
        return np.stack([b[:,0], b[:,1], b[:,2], b[:,3]], axis=1)  # already in image coords
    invs.append(inv_scale11)

    # --- YOLO on each view
    box_sets, score_sets, label_sets = [], [], []
    for img, inv in zip(views, invs):
        r = model.predict(source=img, conf=DET_BASE_CONF, iou=NMS_IOU, verbose=False,
                          device=0 if torch.cuda.is_available() else None)[0]
        if r.boxes is None or r.boxes.xyxy is None or len(r.boxes) == 0:
            box_sets.append(np.zeros((0,4), np.float32))
            score_sets.append(np.zeros((0,), np.float32))
            label_sets.append(np.zeros((0,), np.int32))
            continue

        b = r.boxes.xyxy.cpu().numpy().astype(np.float32)
        s = r.boxes.conf.cpu().numpy().astype(np.float32)
        c = r.boxes.cls.cpu().numpy().astype(np.int32) if r.boxes.cls is not None else np.zeros(len(s), np.int32)
        # invert TTA transform back to original coords
        b = inv(b)
        # clip
        b[:,0] = np.clip(b[:,0], 0, W-1)
        b[:,1] = np.clip(b[:,1], 0, H-1)
        b[:,2] = np.clip(b[:,2], 0, W-1)
        b[:,3] = np.clip(b[:,3], 0, H-1)
        box_sets.append(b); score_sets.append(s); label_sets.append(c)

    # --- WBF fuse
    fboxes, fscores, flabels = _wbf(box_sets, score_sets, label_sets, iou_thr=WBF_IOU, skip_conf=WBF_SKIP_CONF)

    # optional class boost
    if names is not None and PRIORITY_CLASSES:
        for i in range(len(flabels)):
            cname = names[int(flabels[i])]
            if cname in PRIORITY_CLASSES:
                fscores[i] = np.clip(fscores[i] + PRIORITY_BOOST, 0, 1)

    # --- Soft-NMS
    if len(fscores):
        keep = _soft_nms(fboxes, fscores, iou_thr=NMS_IOU, sigma=SOFT_NMS_SIGMA, score_thr=0.01)
        fboxes, fscores, flabels = fboxes[keep], fscores[keep], flabels[keep]

    # --- Hysteresis display thresholding — return only strong boxes to downstream
    out = []
    if names is None:
        names = {}
    for b, s, c in zip(fboxes, fscores, flabels):
        if s >= SHOW_THRESH:
            x1,y1,x2,y2 = b.astype(int)
            if allowed_classes:
                cname = names.get(int(c), str(int(c)))
                if cname not in allowed_classes: 
                    continue
            out.append((x1,y1,x2,y2, float(s), names.get(int(c), str(int(c)))))

    return out


In [1]:
# === Robust YOLO setup: auto-locate best.pt or fall back ===
import os, glob, time, torch
from ultralytics import YOLO

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1) Candidate patterns where Ultralytics saves weights
PROJECT_ROOT = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo"
candidates = [
    rf"{PROJECT_ROOT}\my_nursing_yolo\weights\best.pt",
    rf"{PROJECT_ROOT}\runs\detect\train\weights\best.pt",
    rf"{PROJECT_ROOT}\runs\detect\train*\weights\best.pt",
    rf"{PROJECT_ROOT}\runs\segment\train*\weights\best.pt",
    rf"{PROJECT_ROOT}\runs\train*\weights\best.pt",
    r".\runs\detect\train\weights\best.pt",
    r".\runs\detect\train*\weights\best.pt",
]

# 2) Expand globs and pick the most recent file if multiple
found = []
for pat in candidates:
    found.extend(glob.glob(pat))
best_path = None
if found:
    best_path = max(found, key=lambda p: os.path.getmtime(p))

# 3) Load model (trained if found, else fallback)
if best_path and os.path.exists(best_path):
    print(f"✅ Found trained weights: {best_path}")
    yolo = YOLO(best_path)
else:
    print("⚠️ Trained weights not found. Falling back to yolov8n.pt")
    yolo = YOLO("yolov8n.pt")  # you can change to another base model

# 4) Allowed classes — use prior value if defined, else set a safe default or None to allow all
try:
    ALLOWED_CLASSES  # keep whatever you set earlier
    print(f"Using existing ALLOWED_CLASSES ({len(ALLOWED_CLASSES)} items).")
except NameError:
    # Set to None to allow all, or list your nursing/hospital classes here
    ALLOWED_CLASSES = {
        "person","wheelchair","walker","bed","chair","table","cabinet",
        "basket_bin","door","stretcher","monitor","cart","bottle","bin","tv"
    }
    print(f"Set ALLOWED_CLASSES to default list ({len(ALLOWED_CLASSES)} items).")

print(f"📦 Model ready on {device}.")


✅ Found trained weights: .\runs\detect\train5\weights\best.pt
Set ALLOWED_CLASSES to default list (15 items).
📦 Model ready on cpu.


In [3]:
import cv2, glob, os
import numpy as np

# Pick your test video and output path
test_video = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp700_avoidance.mp4"
save_path   = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\outputs\videos\exp700_boosted_preview.mp4"

cap = cv2.VideoCapture(test_video)
fps = cap.get(cv2.CAP_PROP_FPS) or 20.0
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(save_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

frame_count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1

    # --- boosted detection ---
    det = yolo_obstacles_boosted(frame, yolo, allowed_classes=ALLOWED_CLASSES or None)
    boxes = [(x1, y1, x2, y2) for (x1, y1, x2, y2, score, cname) in det]

    # --- visualization ---
    vis = frame.copy()
    for (x1, y1, x2, y2, score, cname) in det:
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(vis, f"{cname}:{score:.2f}", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(vis, f"Frame {frame_count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    writer.write(vis)

cap.release()
writer.release()
print(f"✅ Boosted inference video saved to → {save_path}")


NameError: name 'yolo_obstacles_boosted' is not defined

In [ ]:
# === CELL: Evaluate current weights on VAL and save per-class metrics ===
from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import numpy as np
import torch, os, sys

# --- Paths (EDIT IF NEEDED) ---
WEIGHTS   = r"C:\Users\ADMIN\hospital\runs\fine_tune_10ep\weights\best.pt"
DATA_YAML = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\data.yaml"
OUT_CSV   = Path(r"C:\Users\ADMIN\hospital\runs\metrics_val_per_class.csv")

# --- Sanity checks ---
if not os.path.isfile(WEIGHTS):
    raise FileNotFoundError(f"Can't find weights at:\n{WEIGHTS}")
if not os.path.isfile(DATA_YAML):
    raise FileNotFoundError(f"Can't find data.yaml at:\n{DATA_YAML}")
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# --- Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Load & validate ---
model = YOLO(WEIGHTS)

# Note: this evaluates the model as-is (no boosted postproc). Use earlier custom PR script for boosted pipeline.
val_results = model.val(
    data=DATA_YAML,
    split="val",          # change to "test" if your data.yaml defines one
    imgsz=640,
    conf=0.001,           # let PR curves decide; keep very low here
    iou=0.6,              # adjust if you want different IoU threshold for metrics
    device=device,
    workers=0,
    plots=False,
    verbose=True
)

# --- Overall metrics (handle key name differences across versions) ---
rd = val_results.results_dict if hasattr(val_results, "results_dict") else {}

def _get(rd, keys, default=np.nan):
    for k in keys:
        if k in rd: return rd[k]
    return default

overall_precision = _get(rd, ["metrics/precision(B)","metrics/precision"], np.nan)
overall_recall    = _get(rd, ["metrics/recall(B)","metrics/recall"], np.nan)
overall_map50     = _get(rd, ["metrics/mAP50(B)","metrics/mAP50"], np.nan)
overall_map5095   = _get(rd, ["metrics/mAP50-95(B)","metrics/mAP50-95"], np.nan)

print("\n=== OVERALL (VAL) ===")
print(f"Precision     : {overall_precision:.4f}")
print(f"Recall        : {overall_recall:.4f}")
print(f"mAP@0.50      : {overall_map50:.4f}")
print(f"mAP@0.50:0.95 : {overall_map5095:.4f}")

# --- Per-class metrics ---
# names: {id: name}
names = getattr(val_results, "names", None)
if names is None and hasattr(model, "model") and hasattr(model.model, "names"):
    names = model.model.names
if isinstance(names, dict):
    class_ids = sorted(int(k) for k in names.keys())
    id_to_name = {int(k): v for k, v in names.items()}
else:
    # fallback
    id_to_name = {i: str(n) for i, n in enumerate(names or [])}
    class_ids = sorted(id_to_name.keys())

# Per-class arrays (may be None depending on version)
maps  = getattr(val_results, "maps", None)                  # mAP@0.50:0.95 per class
ap50  = getattr(getattr(val_results, "box", None), "ap50", None)  # mAP@0.50 per class
prec  = getattr(getattr(val_results, "box", None), "p", None)     # precision summary per class
rec   = getattr(getattr(val_results, "box", None), "r", None)     # recall summary per class

# Normalize shapes & fallbacks
num_classes = len(class_ids)
def _safe_arr(a, fill=np.nan):
    if a is None: return np.full((num_classes,), fill, dtype=float)
    a = np.array(a, dtype=float)
    if a.shape[0] != num_classes:
        # try to align by min length
        b = np.full((num_classes,), fill, dtype=float)
        n = min(num_classes, a.shape[0])
        b[:n] = a[:n]
        return b
    return a

maps = _safe_arr(maps)
ap50 = _safe_arr(ap50)
prec = _safe_arr(prec)
rec  = _safe_arr(rec)

df = pd.DataFrame({
    "class_id": class_ids,
    "class_name": [id_to_name.get(i, str(i)) for i in class_ids],
    "precision": prec,
    "recall": rec,
    "mAP50": ap50,
    "mAP50_95": maps
}).sort_values("class_id").reset_index(drop=True)

df.to_csv(OUT_CSV, index=False)
print(f"\n📄 Per-class metrics saved to: {OUT_CSV}")


In [4]:
from ultralytics import YOLO
import torch, os, shutil

# ---- YOUR PATHS (edit if needed) ----
DATA_YAML = r"C:\Users\ADMIN\Desktop\hospital_avoidance_demo\datasets\nursing_yolo\data.yaml"
PREV_BEST = r"C:\Users\ADMIN\hospital\runs\fine_tune_10ep\weights\best.pt"  # your current best
RUN_DIR   = r"C:\Users\ADMIN\hospital\runs\fine_tune_90plus"

os.makedirs(RUN_DIR, exist_ok=True)

# Try a larger backbone for a final push (comment to keep your current)
# Options: "yolov8m.pt" / "yolov8l.pt" / "yolov8x.pt" (pick one your GPU can handle)
BASE_BACKBONE = "yolov8m.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Start from the larger backbone but warm-start with your head (faster/better than training from scratch)
# If you prefer to strictly continue from your previous .pt, set START_WEIGHTS = PREV_BEST
START_WEIGHTS = BASE_BACKBONE


Device: cpu


In [5]:
# High-quality training overrides (8.3.x–safe)
OVERRIDES = dict(
    data=DATA_YAML,
    imgsz=768,                # bigger image → better P/R
    epochs=30,                # bump to 50 if you can
    batch=16,                 # lower if VRAM is tight
    device=0 if device=="cuda" else "cpu",

    # Optim & LR schedule
    optimizer="AdamW",
    lr0=0.003,                # slightly higher initial LR
    lrf=0.05,                 # cosine min-lr factor
    cos_lr=True,
    momentum=0.9,
    weight_decay=0.0005,

    # Loss weights (kept standard-compatible)
    box=7.5,                  # a touch more box weight boosts precision
    cls=1.2,
    dfl=1.0,

    # Data augmentation (safe for indoor scenes)
    hsv_h=0.012, hsv_s=0.6, hsv_v=0.4,
    degrees=5.0, translate=0.08, scale=0.40, shear=2.0,
    perspective=0.000, flipud=0.0, fliplr=0.25,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.20,
    # (Some builds don’t support 'erasing' or 'ema' as args — omitted)

    # Train behavior
    patience=15,              # early-stop patience
    cache=True,
    amp=True,                 # mixed precision
    rect=True,                # rectangular batches can stabilize indoor layouts
    workers=0,

    project=RUN_DIR,
    name="finetune_90plus"
)
print("Overrides (compatible) ready.")


Overrides (compatible) ready.


In [ ]:
from ultralytics import YOLO

model = YOLO(START_WEIGHTS)
print("Starting fine-tune…")
result = model.train(**OVERRIDES, resume=False)

print("Done. Best weights:", model.ckpt_path if hasattr(model, "ckpt_path") else "see runs dir")


In [ ]:
# Pick the best.pt we just produced
BEST = os.path.join(RUN_DIR, "finetune_90plus", "weights", "best.pt")
if not os.path.exists(BEST):
    # Fallback: search
    for root,_,files in os.walk(RUN_DIR):
        for f in files:
            if f == "best.pt":
                BEST = os.path.join(root, f)

print("Validating:", BEST)

m = YOLO(BEST)
val = m.val(
    data=DATA_YAML,
    split="val",         # change to "test" if you have test split
    imgsz=768,
    conf=0.001,
    iou=0.6,
    device=0 if device=="cuda" else "cpu",
    workers=0,
    augment=True,        # TTA in val to squeeze extra %
    verbose=True
)

d = val.results_dict
print("\n=== TARGET METRICS (VAL+TTA) ===")
print(f"Precision   : {d['metrics/precision(B)']:.4f}")
print(f"Recall      : {d['metrics/recall(B)']:.4f}")
print(f"mAP@0.50    : {d['metrics/mAP50(B)']:.4f}")
print(f"mAP@0.50:0.95: {d['metrics/mAP50-95(B)']:.4f}")
